# CS-781 Phase 6 — E8 repetition-controlled decoding (Architecture C / E7 rank-16 LoRA)

**Purpose:** reuse the already-trained E7 Architecture-C rank-16 LoRA adapter and improve only autoregressive decoding. No gradient training is performed.

This notebook evaluates four deterministic decoding configurations on a small, fixed held-out validation subset:
1. Original greedy decoding (baseline)
2. Mild repetition penalty + 4-gram blocking
3. Stronger repetition penalty + 4-gram blocking
4. Mild repetition penalty + 3-gram blocking

It compares validation BERTScore F1 (primary), ROUGE-L / ROUGE-1, and sentence-repetition rate, then generates the official test predictions with the selected configuration. It writes a **portal-ready ZIP containing only root-level `prediction.json`**.

## Before running

1. Attach the same Phase-1 Kaggle inputs used for E7: Architecture C / DMLC-150 checkpoint and the exact `LEGAL_40K` tokenizer.
2. Enable Kaggle Internet so the official `phase_6` dataset can be loaded.
3. **Reuse the E7 adapter already saved in Kaggle Outputs. No manual upload is needed when it is available to this kernel.**
   - The notebook searches first for `/kaggle/working/CS781_PHASE6_HIGH_SCORE/experiments/E7_C_LoRA_R16_FullData/best_adapter.pt`, then recursively searches `/kaggle/working` and `/kaggle/input` for `best_adapter.pt`.
   - If you are continuing in the still-running E7 notebook/kernel, run these E8 cells in that same notebook session so `/kaggle/working` remains available. Do not restart that kernel first.
   - If you open E8 as a **separate Kaggle notebook**, attach the E7 notebook's saved output through Kaggle's **Add Input / Notebook Output** flow. The recursive search will find `best_adapter.pt` under `/kaggle/input`. You do not need to create a separate dataset manually.
   - `E7_C_LoRA.zip` is only the prediction archive; it is not the adapter.
4. Select a GPU accelerator. This notebook does not retrain E7.

The validation search uses 12 examples × 4 settings to conserve the remaining runtime. This is a quick selection experiment, not a statistically precise benchmark. Allow roughly 1–2 hours depending on Kaggle GPU speed and generation length. The notebook reports the chosen setting and validation metrics; it does not guarantee a higher leaderboard score.

In [ ]:
import os, sys, json, re, gc, math, time, random, hashlib, zipfile, shutil, warnings, traceback, inspect
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint
from tokenizers import Tokenizer
from IPython.display import display, FileLink

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
SEED = 781
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
IS_CUDA = DEVICE.type == "cuda"
print("Python:", sys.version.split()[0], "| PyTorch:", torch.__version__, "| device:", DEVICE)
if IS_CUDA:
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"GPU {i}: {p.name} | VRAM={p.total_memory/1e9:.2f} GB")

ROOT = Path("/kaggle/working/CS781_PHASE6_HIGH_SCORE")
DATA_DIR = ROOT / "data"
EXP_DIR = ROOT / "experiments"
SUB_DIR = ROOT / "submissions"
CACHE_DIR = ROOT / "token_cache"
for d in (ROOT, DATA_DIR, EXP_DIR, SUB_DIR, CACHE_DIR):
    d.mkdir(parents=True, exist_ok=True)

OFFICIAL_REPO_ID = "Exploration-Lab/CS-781-Capstone"
TOKENIZER_SHA256 = "3e38dd5c561d84ad8fbadf792bbca490f8c70346d922df7afbef37c5a942b71e"
ARCH_CLASS_NAMES = {"A":"LegalDecoderA", "B":"LegalHybrid150", "C":"DMLC150", "D":"LegalPrefixFormer145"}
MAX_SOURCE_TOKENS = 768
MAX_TARGET_TOKENS = 160
MAX_TRAIN_EXAMPLES = None       # use the entire public training split after validation holdout
MAX_STEPS_PER_EXPERIMENT = 22000  # capped long-run LoRA fine-tuning steps
VAL_EXAMPLES = 300
VAL_EVAL_LIMIT = 64            # larger held-out validation subset for checkpoint selection
MAX_MINUTES_PER_EXPERIMENT = 210
GRAD_ACCUM = 1
LEARNING_RATE = 2e-5
SAVE_EVERY = 500
GEN_MAX_NEW_TOKENS = 160
GENERATION_TEMPERATURE = 0.0
TOTAL_BUDGET_MINUTES = 270      # includes setup, long training, validation, and test generation
SESSION_START = time.time()
print("Output root:", ROOT)
print("Training budget:", TOTAL_BUDGET_MINUTES, "minutes | per-experiment cap:", MAX_MINUTES_PER_EXPERIMENT, "minutes")
if not IS_CUDA: print("WARNING: GPU not detected. Select a Kaggle GPU accelerator before running the six experiments.")


In [ ]:

def get_hf_token():
    token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")
    if token: return token
    try:
        from kaggle_secrets import UserSecretsClient
        cli = UserSecretsClient()
        for key in ("HF", "HF_TOKEN"):
            try:
                v = cli.get_secret(key)
                if v and v.strip(): return v.strip()
            except Exception: pass
    except Exception: pass
    return None

HF_TOKEN = get_hf_token()
SKIP_DIRS = {"__pycache__", ".git", ".cache", "node_modules", "lost+found", "token_cache", "token_cache_v1", "corpus_cache"}

def iter_files(base, suffixes=None, names=None):
    base = Path(base)
    if not base.exists(): return
    suff = {s.lower() for s in suffixes} if suffixes else None
    nms = {s.lower() for s in names} if names else None
    for root, dirs, files in os.walk(base):
        dirs[:] = [d for d in dirs if d.lower() not in SKIP_DIRS]
        for fn in files:
            p = Path(root) / fn
            if suff and p.suffix.lower() not in suff: continue
            if nms and fn.lower() not in nms: continue
            yield p

def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(4*1024*1024), b""): h.update(b)
    return h.hexdigest()

# This is a clean sixth-run workspace; existing five submissions are not imported or modified.

# Exact Phase-6 layout. Do not accidentally use another phase's train/test files.
PHASE6_EXPECTED = {
    "train": "Phase_6_Summ/Train/train.json",
    "test": "Phase_6_Summ/Test/test.json",
}
DATA_PATHS = {}
for p in iter_files("/kaggle/input", suffixes={".json"}, names={"train.json", "test.json"}):
    low = str(p).replace("\\\\", "/").lower()
    if "/phase_6_summ/" not in low and "/phase6_summ/" not in low: continue
    if low.endswith("/phase_6_summ/train/train.json") or low.endswith("/phase6_summ/train/train.json"): DATA_PATHS["train"] = p
    elif low.endswith("/phase_6_summ/test/test.json") or low.endswith("/phase6_summ/test/test.json"): DATA_PATHS["test"] = p

missing = [k for k in PHASE6_EXPECTED if k not in DATA_PATHS]
HF_SPLITS_LOADED = {}

# The course PDF explicitly maps the Hugging Face configuration `phase_6`
# to the Phase_6_Summ data (repository: Exploration-Lab/CS-781-Capstone).
# Prefer the published configuration; raw-file discovery is a fallback.
if missing:
    print("Phase-6 files not attached locally:", missing)
    print(f"Trying official Hugging Face configuration: {OFFICIAL_REPO_ID!r}, config='phase_6'")
    try:
        from datasets import load_dataset
        try:
            hf_phase6 = load_dataset(OFFICIAL_REPO_ID, "phase_6", token=HF_TOKEN)
        except TypeError:
            # Older datasets versions may not accept the token keyword.
            hf_phase6 = load_dataset(OFFICIAL_REPO_ID, "phase_6", use_auth_token=HF_TOKEN)
        print("Hugging Face Phase-6 splits:", list(hf_phase6.keys()) if hasattr(hf_phase6, "keys") else type(hf_phase6).__name__)

        def find_hf_split(ds, wanted):
            aliases = {"train": {"train", "training"}, "test": {"test", "testing"}}
            if not hasattr(ds, "keys"):
                return None
            for split_name in ds.keys():
                if str(split_name).lower() in aliases[wanted]:
                    return ds[split_name]
            return None

        for kind in list(missing):
            split = find_hf_split(hf_phase6, kind)
            if split is None:
                continue
            out_path = DATA_DIR / f"hf_phase_6_{kind}.json"
            rows = [dict(row) for row in split]
            out_path.write_text(json.dumps(rows, ensure_ascii=False), encoding="utf-8")
            DATA_PATHS[kind] = out_path
            HF_SPLITS_LOADED[kind] = len(rows)
            print(f"Loaded HF config split {kind!r}: {len(rows):,} rows -> {out_path}")
    except Exception as e:
        print("Hugging Face config loading did not succeed; trying repository file listing.")
        print(type(e).__name__ + ":", str(e)[:500])

# Fallback: the PDF also documents the physical files under Phase_6_Summ/Train
# and Phase_6_Summ/Test. Download those exact JSON files from the same repo.
missing = [k for k in PHASE6_EXPECTED if k not in DATA_PATHS]
if missing:
    try:
        from huggingface_hub import HfApi, hf_hub_download
        api = HfApi(token=HF_TOKEN)
        files = api.list_repo_files(repo_id=OFFICIAL_REPO_ID, repo_type="dataset")
        norm = lambda x: x.replace("\\", "/").lower()
        for kind in missing:
            suffix = norm(PHASE6_EXPECTED[kind])
            matches = [f for f in files if norm(f).endswith(suffix)]
            if not matches:
                matches = [f for f in files if "phase_6_summ" in norm(f) and norm(f).endswith("/" + kind + ".json")]
            if not matches:
                # Keep exact phase/config scoping; never substitute another phase's train/test file.
                raise FileNotFoundError(f"No Phase_6_Summ {kind}.json file found in {OFFICIAL_REPO_ID}.")
            matches.sort(key=lambda x: (0 if norm(x) == suffix else 1, len(x), x))
            chosen = matches[0]
            DATA_PATHS[kind] = Path(hf_hub_download(
                repo_id=OFFICIAL_REPO_ID, filename=chosen, repo_type="dataset", token=HF_TOKEN
            ))
            print("Downloaded official Phase-6 file:", chosen, "->", DATA_PATHS[kind])
    except Exception as e:
        raise RuntimeError(
            "Could not obtain Phase-6 data. The course PDF identifies the repository "
            f"{OFFICIAL_REPO_ID} and config 'phase_6' (mapped to Phase_6_Summ). "
            "Enable Kaggle Internet; if the repository is gated, add Kaggle Secret HF or HF_TOKEN. "
            "Do not attach Phase-5 JSON as a substitute. Original error: " + repr(e)
        ) from e

missing = [k for k in PHASE6_EXPECTED if k not in DATA_PATHS]
if missing:
    raise FileNotFoundError(f"Phase-6 data still missing after Hugging Face loading: {missing}")

def unwrap_records(raw):
    for _ in range(6):
        if isinstance(raw, dict):
            k = next((k for k in ("data","records","examples","items","train","test","dataset","instances") if k in raw and isinstance(raw[k], (list,dict))), None)
            if k is not None: raw = raw[k]; continue
        break
    rows = []
    if isinstance(raw, list): rows = raw
    elif isinstance(raw, dict):
        for k,v in raw.items():
            if isinstance(v, dict):
                q = dict(v); q.setdefault("id", k); rows.append(q)
    out = []
    for x in rows:
        if not isinstance(x, dict): continue
        low = {str(k).lower():k for k in x}
        def pick(names):
            return next((x[low[n]] for n in names if n in low), None)
        rid = pick(("id","doc_id","document_id","case_id","uid"))
        doc = pick(("document","text","content","case_text","judgment","judgement"))
        summ = pick(("summary","reference_summary","target","abstract"))
        if rid is None or not isinstance(doc,str) or not doc.strip(): continue
        row = {"id":str(rid), "document":doc}
        if summ is not None and isinstance(summ,str): row["summary"] = summ.strip()
        out.append(row)
    if not out: raise ValueError("Could not parse Phase-6 JSON. Expected id/document/summary fields.")
    ids = [x["id"] for x in out]
    if len(ids) != len(set(ids)): raise ValueError("Duplicate IDs detected in Phase-6 split.")
    return out

def load_json(path):
    with open(path, "r", encoding="utf-8") as f: return unwrap_records(json.load(f))

TRAIN_ALL = load_json(DATA_PATHS["train"])
TEST_RECORDS = load_json(DATA_PATHS["test"])
if any("summary" not in x for x in TRAIN_ALL): raise ValueError("Phase-6 train records must include reference summary.")
rng = np.random.default_rng(SEED)
indices = rng.permutation(len(TRAIN_ALL))
nval = max(1, min(VAL_EXAMPLES, int(0.05*len(TRAIN_ALL))))
VAL_RECORDS = [TRAIN_ALL[int(i)] for i in indices[:nval]]
TRAIN_RECORDS = [TRAIN_ALL[int(i)] for i in indices[nval:]]
if MAX_TRAIN_EXAMPLES is not None: TRAIN_RECORDS = TRAIN_RECORDS[:MAX_TRAIN_EXAMPLES]
print(f"Phase-6 train={len(TRAIN_RECORDS):,} | validation={len(VAL_RECORDS):,} | test={len(TEST_RECORDS):,}")

# Discover Phase-1 checkpoints by path hints first, then unmistakable config signatures.
def arch_hint(p):
    s = str(p).lower().replace("_", "-").replace("\\", "/")
    if any(k in s for k in ("legalprefixformer", "prefixformer", "arch-d", "arch d", "architecture-d", "master-d", "d-arch")): return "D"
    if any(k in s for k in ("dmlc-150", "dmlc150", "c-arch", "arch-c", "arch c", "architecture-c", "master-c")): return "C"
    if any(k in s for k in ("legalhybrid150", "arch-b", "arch b", "architecture-b", "master-b", "b-arch")): return "B"
    if any(k in s for k in ("legaldecodera", "arch-a", "arch a", "architecture-a", "master-a", "a-arch", "checkpoints-archa")): return "A"
    return None

def get_state(payload):
    for k in ("model_state_dict", "state_dict", "model_weights", "weights", "model"):
        v = payload.get(k) if isinstance(payload, dict) else None
        if isinstance(v, dict) and v and any(torch.is_tensor(z) for z in v.values()): return v
    if isinstance(payload, dict) and payload and all(torch.is_tensor(v) for v in payload.values()): return payload
    return None

def get_cfg(payload):
    for k in ("model_config", "MODEL_CONFIG", "config", "cfg", "model_cfg", "model_config_dict"):
        v = payload.get(k) if isinstance(payload, dict) else None
        if isinstance(v, dict) and ("vocab_size" in v or "d_model" in v): return dict(v)
    return None

def infer_arch_from_cfg(cfg):
    if not isinstance(cfg, dict): return None
    keys = set(cfg)
    # These are architecture-specific fields in the saved Phase-1 configs.
    if "prefix_length" in keys and ("qk_norm" in keys or "activation_checkpointing" in keys): return "D"
    if "local_window" in keys and ("local_query_chunk" in keys or "global_attention_layers" in keys): return "B"
    if any(k in keys for k in ("evidence_encoder_layers", "case_encoder_layers", "encoder_layers", "decoder_layers")): return "C"
    return None

CANDIDATES = {a: [] for a in "ABCD"}
CHECKPOINTS, CHECKPOINT_META = {}, {}
allowed_names = {"best_model.pt", "latest.pt", "model.pt", "checkpoint.pt", "best.pt", "checkpoint.pth", "model.pth"}
for p in iter_files("/kaggle/input", suffixes={".pt", ".pth", ".ckpt"}):
    if p.name.lower() not in allowed_names: continue
    hint = arch_hint(p)
    try:
        try: payload = torch.load(p, map_location="cpu", weights_only=False, mmap=True)
        except Exception: payload = torch.load(p, map_location="cpu", weights_only=False)
        cfg, state = get_cfg(payload), get_state(payload)
        inferred = infer_arch_from_cfg(cfg)
        arch = hint or inferred
        if arch and cfg and state is not None:
            CANDIDATES[arch].append((p, cfg, int(payload.get("step", payload.get("global_step", 0)) or 0)))
        del payload, state
    except Exception as e:
        print(f"Checkpoint candidate rejected: {p} | {type(e).__name__}: {str(e)[:160]}")

for a in "ABCD":
    def score_candidate(item):
        p = item[0]
        nm = 0 if p.name.lower() == "best_model.pt" else 1 if p.name.lower() in {"model.pt", "best.pt"} else 2
        # Prefer named architecture directories over generic recovery output paths.
        explicit = 0 if arch_hint(p) == a else 1
        return (explicit, nm, len(str(p)), str(p))
    if CANDIDATES[a]:
        chosen, cfg, step = sorted(CANDIDATES[a], key=score_candidate)[0]
        CHECKPOINTS[a] = str(chosen)
        CHECKPOINT_META[a] = {"path": str(chosen), "cfg": cfg, "step": step}
    print(f"Architecture {a}: {'FOUND ' + CHECKPOINTS[a] if a in CHECKPOINTS else 'MISSING'}")

# Locate exact 40K Phase-1 tokenizer.
TOK_PATH = None
for p in iter_files("/kaggle/input", names={"tokenizer.json"}):
    try:
        if sha256_file(p) == TOKENIZER_SHA256:
            TOK_PATH = p; break
    except Exception: pass
if TOK_PATH is None:
    raise FileNotFoundError("Exact Phase-1 LEGAL_40K tokenizer was not found. Attach the Phase-1 dataset containing tokenizers/legal_bpe_40k/tokenizer.json.")
TOKENIZER = Tokenizer.from_file(str(TOK_PATH))
assert TOKENIZER.get_vocab_size() == 40000 and sha256_file(TOK_PATH) == TOKENIZER_SHA256
vocab = TOKENIZER.get_vocab()
EOS_ID = next((int(vocab[k]) for k in ("</s>","<|endoftext|>","<eos>","[EOS]") if k in vocab), None)
BOS_ID = next((int(vocab[k]) for k in ("<s>","<|beginoftext|>","<bos>","[BOS]") if k in vocab), EOS_ID if EOS_ID is not None else 0)
PAD_ID = 0
print("Tokenizer:", TOK_PATH, "| vocab=40,000 | EOS:", EOS_ID, "| BOS:", BOS_ID)
print("Compatible architecture inputs:", sorted(CHECKPOINTS), "| missing:", sorted(set("ABCD")-set(CHECKPOINTS)))
(ROOT/"phase6_input_manifest.json").write_text(json.dumps({
    "train_path":str(DATA_PATHS["train"]), "test_path":str(DATA_PATHS["test"]),
    "train_records":len(TRAIN_ALL),"validation_records":len(VAL_RECORDS),"test_records":len(TEST_RECORDS),
    "checkpoint_paths":CHECKPOINTS,"missing_architectures":sorted(set("ABCD")-set(CHECKPOINTS)),
    "tokenizer_path":str(TOK_PATH),"tokenizer_sha256":TOKENIZER_SHA256
}, indent=2), encoding="utf-8")


In [ ]:
# Namespaces are isolated because all four implementation files reuse names such as RMSNorm and SwiGLU.
ARCH_CLASS_NAMES = {"A":"LegalDecoderA", "B":"LegalHybrid150", "C":"DMLC150", "D":"LegalPrefixFormer145"}
ARCH_SOURCE = {}
ARCH_SOURCE['A'] = 'class RMSNorm(nn.Module):\n\n    def __init__(\n        self,\n        dim,\n        eps=1e-5\n    ):\n\n        super().__init__()\n\n        self.weight = nn.Parameter(\n            torch.ones(dim)\n        )\n\n        self.eps = eps\n\n    def forward(self, x):\n\n        inv_rms = torch.rsqrt(\n            x.pow(2).mean(\n                dim=-1,\n                keepdim=True\n            )\n            + self.eps\n        )\n\n        return (\n            x\n            * inv_rms\n            * self.weight\n        )\n\nclass RotaryEmbedding(nn.Module):\n\n    def __init__(\n        self,\n        dim,\n        max_seq_len,\n        theta=10000.0\n    ):\n\n        super().__init__()\n\n        if dim % 2:\n            raise ValueError(\n                "RoPE dimension must be even."\n            )\n\n        inv_freq = 1.0 / (\n            theta ** (\n                torch.arange(\n                    0,\n                    dim,\n                    2,\n                    dtype=torch.float32\n                )\n                / dim\n            )\n        )\n\n        positions = torch.arange(\n            max_seq_len,\n            dtype=torch.float32\n        )\n\n        freqs = torch.outer(\n            positions,\n            inv_freq\n        )\n\n        self.register_buffer(\n            "cos_cached",\n            torch.repeat_interleave(\n                freqs.cos(),\n                2,\n                dim=-1\n            ),\n            persistent=False\n        )\n\n        self.register_buffer(\n            "sin_cached",\n            torch.repeat_interleave(\n                freqs.sin(),\n                2,\n                dim=-1\n            ),\n            persistent=False\n        )\n\n    def forward(\n        self,\n        x,\n        position_offset=0\n    ):\n\n        seq_len = x.size(-2)\n\n        end = (\n            position_offset\n            + seq_len\n        )\n\n        if (\n            end\n            > self.cos_cached.size(0)\n        ):\n            raise ValueError(\n                "RoPE context exceeded."\n            )\n\n        cos = self.cos_cached[\n            position_offset:end\n        ].to(\n            device=x.device,\n            dtype=x.dtype\n        )\n\n        sin = self.sin_cached[\n            position_offset:end\n        ].to(\n            device=x.device,\n            dtype=x.dtype\n        )\n\n        cos = cos.view(\n            1,\n            1,\n            seq_len,\n            -1\n        )\n\n        sin = sin.view(\n            1,\n            1,\n            seq_len,\n            -1\n        )\n\n        even = x[..., 0::2]\n        odd = x[..., 1::2]\n\n        rotated = torch.stack(\n            [\n                -odd,\n                even\n            ],\n            dim=-1\n        ).flatten(-2)\n\n        return (\n            x * cos\n            + rotated * sin\n        )\n\nclass GQAAttention(nn.Module):\n\n    def __init__(\n        self,\n        cfg\n    ):\n\n        super().__init__()\n\n        d_model = cfg[\n            "d_model"\n        ]\n\n        n_heads = cfg[\n            "n_heads"\n        ]\n\n        n_kv_heads = cfg[\n            "n_kv_heads"\n        ]\n\n        head_dim = (\n            d_model // n_heads\n        )\n\n        self.n_heads = n_heads\n        self.n_kv_heads = n_kv_heads\n        self.head_dim = head_dim\n\n        self.group_size = (\n            n_heads\n            // n_kv_heads\n        )\n\n        self.q_proj = nn.Linear(\n            d_model,\n            n_heads * head_dim,\n            bias=cfg["use_bias"]\n        )\n\n        self.k_proj = nn.Linear(\n            d_model,\n            n_kv_heads * head_dim,\n            bias=cfg["use_bias"]\n        )\n\n        self.v_proj = nn.Linear(\n            d_model,\n            n_kv_heads * head_dim,\n            bias=cfg["use_bias"]\n        )\n\n        self.o_proj = nn.Linear(\n            d_model,\n            d_model,\n            bias=cfg["use_bias"]\n        )\n\n        self.rope = RotaryEmbedding(\n            head_dim,\n            cfg["context_length"],\n            cfg["rope_theta"]\n        )\n\n    def forward(\n        self,\n        x,\n        past_kv=None,\n        use_cache=False\n    ):\n\n        batch, seq_len, d_model = (\n            x.shape\n        )\n\n        past_len = (\n            0\n            if past_kv is None\n            else past_kv[0].size(2)\n        )\n\n        q = self.q_proj(\n            x\n        ).view(\n            batch,\n            seq_len,\n            self.n_heads,\n            self.head_dim\n        ).transpose(\n            1,\n            2\n        )\n\n        k = self.k_proj(\n            x\n        ).view(\n            batch,\n            seq_len,\n            self.n_kv_heads,\n            self.head_dim\n        ).transpose(\n            1,\n            2\n        )\n\n        v = self.v_proj(\n            x\n        ).view(\n            batch,\n            seq_len,\n            self.n_kv_heads,\n            self.head_dim\n        ).transpose(\n            1,\n            2\n        )\n\n        q = self.rope(\n            q,\n            past_len\n        )\n\n        k = self.rope(\n            k,\n            past_len\n        )\n\n        if past_kv is not None:\n\n            past_k, past_v = past_kv\n\n            k = torch.cat(\n                [past_k, k],\n                dim=2\n            )\n\n            v = torch.cat(\n                [past_v, v],\n                dim=2\n            )\n\n        if (\n            self.n_kv_heads\n            != self.n_heads\n        ):\n\n            k_attn = (\n                k.repeat_interleave(\n                    self.group_size,\n                    dim=1\n                )\n            )\n\n            v_attn = (\n                v.repeat_interleave(\n                    self.group_size,\n                    dim=1\n                )\n            )\n\n        else:\n\n            k_attn = k\n            v_attn = v\n\n        if past_kv is None:\n\n            y = F.scaled_dot_product_attention(\n                q,\n                k_attn,\n                v_attn,\n                attn_mask=None,\n                dropout_p=0.0,\n                is_causal=True,\n            )\n\n        else:\n\n            y = F.scaled_dot_product_attention(\n                q,\n                k_attn,\n                v_attn,\n                attn_mask=None,\n                dropout_p=0.0,\n                is_causal=False,\n            )\n\n        y = y.transpose(\n            1,\n            2\n        ).contiguous().view(\n            batch,\n            seq_len,\n            d_model\n        )\n\n        output = self.o_proj(\n            y\n        )\n\n        if use_cache:\n            return output, (k, v)\n\n        return output\n\nclass SwiGLU(nn.Module):\n\n    def __init__(\n        self,\n        cfg\n    ):\n\n        super().__init__()\n\n        d_model = cfg[\n            "d_model"\n        ]\n\n        ffn_dim = cfg[\n            "ffn_dim"\n        ]\n\n        self.gate_proj = nn.Linear(\n            d_model,\n            ffn_dim,\n            bias=cfg["use_bias"]\n        )\n\n        self.up_proj = nn.Linear(\n            d_model,\n            ffn_dim,\n            bias=cfg["use_bias"]\n        )\n\n        self.down_proj = nn.Linear(\n            ffn_dim,\n            d_model,\n            bias=cfg["use_bias"]\n        )\n\n    def forward(self, x):\n\n        return self.down_proj(\n            F.silu(\n                self.gate_proj(x)\n            )\n            * self.up_proj(x)\n        )\n\nclass DecoderBlock(nn.Module):\n\n    def __init__(\n        self,\n        cfg\n    ):\n\n        super().__init__()\n\n        d_model = cfg[\n            "d_model"\n        ]\n\n        self.norm1 = RMSNorm(\n            d_model,\n            cfg["norm_eps"]\n        )\n\n        self.attn = GQAAttention(\n            cfg\n        )\n\n        self.norm2 = RMSNorm(\n            d_model,\n            cfg["norm_eps"]\n        )\n\n        self.ffn = SwiGLU(\n            cfg\n        )\n\n    def forward(\n        self,\n        x,\n        past_kv=None,\n        use_cache=False\n    ):\n\n        if use_cache:\n\n            attn_out, present = (\n                self.attn(\n                    self.norm1(x),\n                    past_kv=past_kv,\n                    use_cache=True,\n                )\n            )\n\n            x = x + attn_out\n\n            x = x + self.ffn(\n                self.norm2(x)\n            )\n\n            return x, present\n\n        x = x + self.attn(\n            self.norm1(x)\n        )\n\n        x = x + self.ffn(\n            self.norm2(x)\n        )\n\n        return x\n\nclass LegalDecoderA(nn.Module):\n\n    def __init__(\n        self,\n        cfg\n    ):\n\n        super().__init__()\n\n        self.cfg = dict(cfg)\n\n        self.token_embedding = nn.Embedding(\n            cfg["vocab_size"],\n            cfg["d_model"]\n        )\n\n        self.blocks = nn.ModuleList([\n            DecoderBlock(cfg)\n            for _ in range(\n                cfg["n_layers"]\n            )\n        ])\n\n        self.final_norm = RMSNorm(\n            cfg["d_model"],\n            cfg["norm_eps"]\n        )\n\n        self.lm_head = nn.Linear(\n            cfg["d_model"],\n            cfg["vocab_size"],\n            bias=False\n        )\n\n        if cfg["tie_embeddings"]:\n\n            self.lm_head.weight = (\n                self.token_embedding.weight\n            )\n\n        self.apply(\n            self._init_weights\n        )\n\n        residual_scale = (\n            1.0\n            / math.sqrt(\n                2.0\n                * cfg["n_layers"]\n            )\n        )\n\n        for block in self.blocks:\n\n            nn.init.normal_(\n                block.attn.o_proj.weight,\n                mean=0.0,\n                std=0.02\n                * residual_scale,\n            )\n\n            nn.init.normal_(\n                block.ffn.down_proj.weight,\n                mean=0.0,\n                std=0.02\n                * residual_scale,\n            )\n\n    def _init_weights(\n        self,\n        module\n    ):\n\n        if isinstance(\n            module,\n            nn.Linear\n        ):\n\n            if module.bias is not None:\n                nn.init.zeros_(\n                    module.bias\n                )\n\n            nn.init.normal_(\n                module.weight,\n                mean=0.0,\n                std=0.02,\n            )\n\n        elif isinstance(\n            module,\n            nn.Embedding\n        ):\n\n            nn.init.normal_(\n                module.weight,\n                mean=0.0,\n                std=0.02\n            )\n\n        elif isinstance(\n            module,\n            RMSNorm\n        ):\n\n            nn.init.ones_(\n                module.weight\n            )\n\n    def forward(\n        self,\n        input_ids,\n        labels=None,\n        use_cache=False,\n        past_key_values=None\n    ):\n\n        x = self.token_embedding(\n            input_ids\n        )\n\n        if use_cache:\n\n            if past_key_values is None:\n\n                past_key_values = [\n                    None\n                    for _ in self.blocks\n                ]\n\n            presents = []\n\n            for block, past in zip(\n                self.blocks,\n                past_key_values\n            ):\n\n                x, present = block(\n                    x,\n                    past_kv=past,\n                    use_cache=True\n                )\n\n                presents.append(\n                    present\n                )\n\n            x = self.final_norm(\n                x\n            )\n\n            logits = self.lm_head(\n                x\n            )\n\n            return {\n                "logits": logits,\n                "loss": None,\n                "past_key_values": presents,\n            }\n\n        for block in self.blocks:\n\n            if (\n                self.training\n                and self.cfg[\n                    "gradient_checkpointing"\n                ]\n            ):\n\n                x = torch.utils.checkpoint.checkpoint(\n                    block,\n                    x,\n                    use_reentrant=False\n                )\n\n            else:\n\n                x = block(x)\n\n        x = self.final_norm(\n            x\n        )\n\n        logits = self.lm_head(\n            x\n        )\n\n        loss = None\n\n        if labels is not None:\n\n            loss = F.cross_entropy(\n                logits.reshape(\n                    -1,\n                    logits.size(-1)\n                ),\n                labels.reshape(\n                    -1\n                )\n            )\n\n        return {\n            "logits": logits,\n            "loss": loss,\n        }'
ARCH_SOURCE['B'] = 'class RMSNorm(nn.Module):\n    def __init__(self, dim, eps=1e-5):\n        super().__init__()\n        self.weight = nn.Parameter(torch.ones(dim))\n        self.eps = eps\n\n    def forward(self, x):\n        # Stable fp32 RMS computation, then explicitly restore the incoming dtype.\n        inv_rms = torch.rsqrt(\n            x.float().pow(2).mean(dim=-1, keepdim=True) + self.eps\n        )\n        scale = self.weight.to(dtype=x.dtype)\n        out = (x * inv_rms.to(dtype=x.dtype)) * scale\n        return out.to(dtype=x.dtype)\n\nclass RotaryEmbedding(nn.Module):\n    def __init__(self, dim, max_seq_len, theta=10_000.0):\n        super().__init__()\n        if dim % 2:\n            raise ValueError("RoPE dimension must be even.")\n        inv_freq = 1.0 / (\n            theta ** (\n                torch.arange(0, dim, 2, dtype=torch.float32) / dim\n            )\n        )\n        positions = torch.arange(max_seq_len, dtype=torch.float32)\n        freqs = torch.outer(positions, inv_freq)\n        emb = torch.cat([freqs, freqs], dim=-1)\n        self.register_buffer("cos_cached", emb.cos(), persistent=False)\n        self.register_buffer("sin_cached", emb.sin(), persistent=False)\n\n    def forward(self, x, start_pos=0):\n        seq_len = x.size(-2)\n        end = start_pos + seq_len\n        if end > self.cos_cached.size(0):\n            raise RuntimeError(\n                f"RoPE request {end} exceeds max context {self.cos_cached.size(0)}"\n            )\n        cos = self.cos_cached[start_pos:end].to(dtype=x.dtype)[None, None, :, :]\n        sin = self.sin_cached[start_pos:end].to(dtype=x.dtype)[None, None, :, :]\n        x1 = x[..., :x.size(-1)//2]\n        x2 = x[..., x.size(-1)//2:]\n        rotated = torch.cat([-x2, x1], dim=-1)\n        return x * cos + rotated * sin\n\nclass HybridGQAAttention(nn.Module):\n    def __init__(self, cfg, is_global):\n        super().__init__()\n        d = cfg["d_model"]\n        h = cfg["n_heads"]\n        kvh = cfg["n_kv_heads"]\n        hd = cfg["head_dim"]\n\n        if d != h * hd:\n            raise ValueError("d_model must equal n_heads * head_dim.")\n        if h % kvh != 0:\n            raise ValueError("n_heads must be divisible by n_kv_heads.")\n\n        self.d_model = d\n        self.n_heads = h\n        self.n_kv_heads = kvh\n        self.head_dim = hd\n        self.group_size = h // kvh\n\n        self.is_global = bool(is_global)\n        self.local_window = int(cfg["local_window"])\n        self.query_chunk = int(cfg["local_query_chunk"])\n\n        self.q_proj = nn.Linear(d, h * hd, bias=cfg["use_bias"])\n        self.k_proj = nn.Linear(d, kvh * hd, bias=cfg["use_bias"])\n        self.v_proj = nn.Linear(d, kvh * hd, bias=cfg["use_bias"])\n        self.o_proj = nn.Linear(d, d, bias=cfg["use_bias"])\n\n        self.rope = RotaryEmbedding(\n            hd,\n            cfg["context_length"],\n            cfg["rope_theta"],\n        )\n\n    def _expand_kv(self, k, v):\n        if self.n_kv_heads == self.n_heads:\n            return k, v\n        k = k.repeat_interleave(self.group_size, dim=1)\n        v = v.repeat_interleave(self.group_size, dim=1)\n        return k, v\n\n    def _local_attention_train(self, q, k_full, v_full, past_len):\n        q_len = q.size(-2)\n        outputs = []\n\n        for start in range(0, q_len, self.query_chunk):\n            end = min(q_len, start + self.query_chunk)\n            curr = end - start\n\n            abs_q_start = past_len + start\n            key_start_abs = max(0, abs_q_start - self.local_window + 1)\n            key_end_abs = past_len + end\n\n            k_slice = k_full[:, :, key_start_abs:key_end_abs, :]\n            v_slice = v_full[:, :, key_start_abs:key_end_abs, :]\n\n            k_len = k_slice.size(-2)\n\n            # Exact causal sliding-window mask for this chunk. Construct only\n            # [query_chunk, local_window] rather than a full [T,T] mask.\n            q_positions = torch.arange(\n                abs_q_start,\n                past_len + end,\n                device=q.device,\n                dtype=torch.long,\n            )[:, None]\n            k_positions = torch.arange(\n                key_start_abs,\n                key_end_abs,\n                device=q.device,\n                dtype=torch.long,\n            )[None, :]\n            mask = (\n                (k_positions <= q_positions)\n                & (k_positions >= (q_positions - self.local_window + 1))\n            )\n            assert mask.shape == (curr, k_len)\n\n            y = F.scaled_dot_product_attention(\n                q[:, :, start:end, :],\n                k_slice,\n                v_slice,\n                attn_mask=mask,\n                dropout_p=0.0,\n                is_causal=False,\n            )\n            outputs.append(y)\n\n        return torch.cat(outputs, dim=-2)\n\n    def _local_attention_cached(self, q, k_full, v_full):\n        # Intended generation path: q_len=1. All retained keys are historical\n        # tokens plus the current token, so no causal mask is needed.\n        keep = min(self.local_window, k_full.size(-2))\n        k_slice = k_full[:, :, -keep:, :]\n        v_slice = v_full[:, :, -keep:, :]\n        return F.scaled_dot_product_attention(\n            q,\n            k_slice,\n            v_slice,\n            attn_mask=None,\n            dropout_p=0.0,\n            is_causal=False,\n        )\n\n    def forward(self, x, past_kv=None, use_cache=False):\n        bsz, q_len, _ = x.shape\n        past_len = 0 if past_kv is None else past_kv[0].size(2)\n\n        if past_len + q_len > self.rope.cos_cached.size(0):\n            raise RuntimeError("Attention request exceeds context length.")\n\n        q = self.q_proj(x).view(\n            bsz, q_len, self.n_heads, self.head_dim\n        ).transpose(1, 2)\n\n        k = self.k_proj(x).view(\n            bsz, q_len, self.n_kv_heads, self.head_dim\n        ).transpose(1, 2)\n\n        v = self.v_proj(x).view(\n            bsz, q_len, self.n_kv_heads, self.head_dim\n        ).transpose(1, 2)\n\n        q = self.rope(q, past_len)\n        k = self.rope(k, past_len)\n\n        if past_kv is not None:\n            past_k, past_v = past_kv\n            k_full = torch.cat([past_k, k], dim=2)\n            v_full = torch.cat([past_v, v], dim=2)\n        else:\n            k_full = k\n            v_full = v\n\n        if self.n_kv_heads != self.n_heads:\n            k_attn, v_attn = self._expand_kv(k_full, v_full)\n        else:\n            k_attn, v_attn = k_full, v_full\n\n        if self.is_global:\n            if past_kv is None:\n                y = F.scaled_dot_product_attention(\n                    q,\n                    k_attn,\n                    v_attn,\n                    attn_mask=None,\n                    dropout_p=0.0,\n                    is_causal=True,\n                )\n            else:\n                if q_len != 1:\n                    raise RuntimeError(\n                        "Cached global attention currently expects q_len=1."\n                    )\n                y = F.scaled_dot_product_attention(\n                    q,\n                    k_attn,\n                    v_attn,\n                    attn_mask=None,\n                    dropout_p=0.0,\n                    is_causal=False,\n                )\n        else:\n            if past_kv is not None:\n                if q_len != 1:\n                    raise RuntimeError(\n                        "Cached local attention currently expects q_len=1."\n                    )\n                y = self._local_attention_cached(q, k_attn, v_attn)\n            else:\n                y = self._local_attention_train(\n                    q, k_attn, v_attn, past_len=0\n                )\n\n        y = y.transpose(1, 2).contiguous().view(\n            bsz, q_len, self.d_model\n        )\n        y = self.o_proj(y)\n\n        if use_cache:\n            return y, (k_full, v_full)\n        return y\n\nclass SwiGLU(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        d = cfg["d_model"]\n        f = cfg["ffn_dim"]\n        self.gate_proj = nn.Linear(d, f, bias=cfg["use_bias"])\n        self.up_proj = nn.Linear(d, f, bias=cfg["use_bias"])\n        self.down_proj = nn.Linear(f, d, bias=cfg["use_bias"])\n\n    def forward(self, x):\n        return self.down_proj(\n            F.silu(self.gate_proj(x)) * self.up_proj(x)\n        )\n\nclass HybridBlock(nn.Module):\n    def __init__(self, cfg, layer_index):\n        super().__init__()\n        self.norm1 = RMSNorm(cfg["d_model"], cfg["norm_eps"])\n        self.attn = HybridGQAAttention(\n            cfg,\n            is_global=((layer_index + 1) % cfg["global_every"] == 0),\n        )\n        self.norm2 = RMSNorm(cfg["d_model"], cfg["norm_eps"])\n        self.ffn = SwiGLU(cfg)\n\n    def forward(self, x, past_kv=None, use_cache=False):\n        if use_cache:\n            attn_out, present = self.attn(\n                self.norm1(x),\n                past_kv=past_kv,\n                use_cache=True,\n            )\n            x = x + attn_out\n            x = x + self.ffn(self.norm2(x))\n            return x, present\n\n        x = x + self.attn(self.norm1(x))\n        x = x + self.ffn(self.norm2(x))\n        return x\n\nclass LegalHybrid150(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        self.cfg = dict(cfg)\n\n        self.token_embedding = nn.Embedding(\n            cfg["vocab_size"], cfg["d_model"]\n        )\n        self.blocks = nn.ModuleList([\n            HybridBlock(cfg, i)\n            for i in range(cfg["n_layers"])\n        ])\n        self.final_norm = RMSNorm(\n            cfg["d_model"], cfg["norm_eps"]\n        )\n        self.lm_head = nn.Linear(\n            cfg["d_model"],\n            cfg["vocab_size"],\n            bias=False,\n        )\n\n        # Initialize parameters once, then tie the LM head.\n        # This avoids double-initializing the same shared Parameter.\n        self.apply(self._init_weights)\n\n        if cfg["tie_embeddings"]:\n            self.lm_head.weight = self.token_embedding.weight\n\n        residual_scale = 1.0 / math.sqrt(2.0 * cfg["n_layers"])\n        for block in self.blocks:\n            nn.init.normal_(\n                block.attn.o_proj.weight,\n                mean=0.0,\n                std=0.02 * residual_scale,\n            )\n            nn.init.normal_(\n                block.ffn.down_proj.weight,\n                mean=0.0,\n                std=0.02 * residual_scale,\n            )\n\n    def _init_weights(self, module):\n        if isinstance(module, nn.Linear):\n            if module.bias is not None:\n                nn.init.zeros_(module.bias)\n            nn.init.normal_(module.weight, mean=0.0, std=0.02)\n        elif isinstance(module, nn.Embedding):\n            nn.init.normal_(module.weight, mean=0.0, std=0.02)\n        elif isinstance(module, RMSNorm):\n            nn.init.ones_(module.weight)\n\n    def forward(\n        self,\n        input_ids,\n        labels=None,\n        use_cache=False,\n        past_key_values=None,\n    ):\n        input_ids = input_ids.long()\n        x = self.token_embedding(input_ids)\n        # Keep the residual stream in FP16 on CUDA under the training/inference\n        # autocast path; this avoids unintended FP32 residual accumulation.\n        if x.is_cuda and x.dtype != torch.float16:\n            x = x.to(dtype=torch.float16)\n\n        if use_cache:\n            if past_key_values is None:\n                past_key_values = [None] * len(self.blocks)\n\n            presents = []\n            for block, past in zip(self.blocks, past_key_values):\n                x, present = block(\n                    x,\n                    past_kv=past,\n                    use_cache=True,\n                )\n                presents.append(present)\n\n            x = self.final_norm(x)\n            logits = self.lm_head(x)\n\n            return {\n                "logits": logits,\n                "loss": None,\n                "past_key_values": presents,\n            }\n\n        for block in self.blocks:\n            if (\n                self.training\n                and self.cfg["gradient_checkpointing"]\n            ):\n                x = torch.utils.checkpoint.checkpoint(\n                    block,\n                    x,\n                    use_reentrant=False,\n                )\n            else:\n                x = block(x)\n\n        x = self.final_norm(x)\n\n        if labels is not None:\n            # Keep peak memory reasonable by projecting in time chunks.\n            labels = labels.long()\n            time_chunk = 512\n            losses = []\n            for start in range(0, x.size(1), time_chunk):\n                end = min(x.size(1), start + time_chunk)\n                logits_chunk = self.lm_head(x[:, start:end, :])\n                loss_chunk = F.cross_entropy(\n                    logits_chunk.reshape(-1, logits_chunk.size(-1)),\n                    labels[:, start:end].reshape(-1),\n                )\n                losses.append(loss_chunk)\n                del logits_chunk\n            loss = torch.stack(losses).mean()\n            return {"loss": loss}\n\n        logits = self.lm_head(x)\n        return {"logits": logits, "loss": None}\n\ndef estimate_model_params(cfg):\n    v = cfg["vocab_size"]\n    d = cfg["d_model"]\n    L = cfg["n_layers"]\n    H = cfg["n_heads"]\n    KV = cfg["n_kv_heads"]\n    hd = cfg["head_dim"]\n    f = cfg["ffn_dim"]\n\n    embedding = v * d\n    attention = (\n        d * H * hd\n        + d * KV * hd\n        + d * KV * hd\n        + (H * hd) * d\n    )\n    ffn = 3 * d * f\n    norms = (2 * L + 1) * d\n    output_head = 0 if cfg["tie_embeddings"] else v * d\n\n    return embedding + L * (attention + ffn) + norms + output_head'
ARCH_SOURCE['C'] = 'class RMSNorm(nn.Module):\n    def __init__(self, dim, eps=1e-5):\n        super().__init__()\n        self.weight = nn.Parameter(torch.ones(dim))\n        self.eps = eps\n\n    def forward(self, x):\n        inv_rms = torch.rsqrt(x.float().pow(2).mean(dim=-1, keepdim=True) + self.eps)\n        return x * inv_rms.to(dtype=x.dtype) * self.weight.to(dtype=x.dtype)\n\nclass RotaryEmbedding(nn.Module):\n    def __init__(self, dim, max_seq_len, theta=10_000.0):\n        super().__init__()\n        if dim % 2:\n            raise ValueError("RoPE dimension must be even.")\n        inv_freq = 1.0 / (theta ** (torch.arange(0, dim, 2, dtype=torch.float32) / dim))\n        positions = torch.arange(max_seq_len, dtype=torch.float32)\n        freqs = torch.outer(positions, inv_freq)\n        emb = torch.cat([freqs, freqs], dim=-1)\n        self.register_buffer("cos_cached", emb.cos(), persistent=False)\n        self.register_buffer("sin_cached", emb.sin(), persistent=False)\n\n    def get(self, start, length, device, dtype):\n        if start + length > self.cos_cached.size(0):\n            raise RuntimeError("Requested RoPE positions exceed context length.")\n        cos = self.cos_cached[start:start+length].to(device=device, dtype=dtype).view(1, 1, length, -1)\n        sin = self.sin_cached[start:start+length].to(device=device, dtype=dtype).view(1, 1, length, -1)\n        return cos, sin\n\ndef rotate_half(x):\n    d = x.size(-1)\n    return torch.cat((-x[..., d//2:], x[..., :d//2]), dim=-1)\n\ndef apply_rope(x, cos, sin):\n    return x * cos + rotate_half(x) * sin\n\nclass GQAAttention(nn.Module):\n    def __init__(self, d_model, q_heads, kv_heads, head_dim, max_seq_len, theta=10_000.0, causal=False, use_rope=True):\n        super().__init__()\n        if d_model != q_heads * head_dim:\n            raise ValueError("d_model must equal q_heads * head_dim")\n        if q_heads % kv_heads:\n            raise ValueError("q_heads must be divisible by kv_heads")\n        self.q_heads = q_heads\n        self.kv_heads = kv_heads\n        self.head_dim = head_dim\n        self.causal = causal\n        self.use_rope = use_rope\n        self.q_proj = nn.Linear(d_model, q_heads * head_dim, bias=False)\n        self.k_proj = nn.Linear(d_model, kv_heads * head_dim, bias=False)\n        self.v_proj = nn.Linear(d_model, kv_heads * head_dim, bias=False)\n        self.o_proj = nn.Linear(q_heads * head_dim, d_model, bias=False)\n        self.rope = RotaryEmbedding(head_dim, max_seq_len, theta) if use_rope else None\n\n    def _q(self, x):\n        return self.q_proj(x).view(x.size(0), x.size(1), self.q_heads, self.head_dim).transpose(1, 2)\n\n    def _kv(self, x):\n        k = self.k_proj(x).view(x.size(0), x.size(1), self.kv_heads, self.head_dim).transpose(1, 2)\n        v = self.v_proj(x).view(x.size(0), x.size(1), self.kv_heads, self.head_dim).transpose(1, 2)\n        return k, v\n\n    def _repeat_kv(self, x):\n        return x.repeat_interleave(self.q_heads // self.kv_heads, dim=1)\n\n    @staticmethod\n    def _mask(mask):\n        if mask is None:\n            return None\n        if mask.dim() == 2:\n            return mask[:, None, None, :]\n        if mask.dim() == 3:\n            return mask[:, None, :, :]\n        return mask\n\n    def forward(self, q_input, kv_input=None, attn_mask=None, cache=None, use_cache=False, position_offset=0):\n        is_cross = kv_input is not None\n        source = q_input if kv_input is None else kv_input\n        q = self._q(q_input)\n        k_new, v_new = self._kv(source)\n\n        if self.use_rope and not is_cross:\n            cos_k, sin_k = self.rope.get(position_offset, k_new.size(2), k_new.device, k_new.dtype)\n            cos_q, sin_q = self.rope.get(position_offset, q.size(2), q.device, q.dtype)\n            k_new = apply_rope(k_new, cos_k, sin_k)\n            q = apply_rope(q, cos_q, sin_q)\n\n        if cache is not None:\n            k_cache = torch.cat([cache[0], k_new], dim=2)\n            v_cache = torch.cat([cache[1], v_new], dim=2)\n        else:\n            k_cache, v_cache = k_new, v_new\n\n        k = self._repeat_kv(k_cache)\n        v = self._repeat_kv(v_cache)\n        mask = self._mask(attn_mask)\n\n        if self.causal and cache is None and mask is None:\n            out = F.scaled_dot_product_attention(q, k, v, dropout_p=0.0, is_causal=True)\n        else:\n            out = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=0.0, is_causal=False)\n\n        out = out.transpose(1, 2).contiguous().view(q_input.size(0), q_input.size(1), -1)\n        out = self.o_proj(out)\n        new_cache = (k_cache, v_cache) if use_cache else None\n        return out, new_cache\n\nclass SwiGLU(nn.Module):\n    def __init__(self, d_model, ffn_dim):\n        super().__init__()\n        self.gate_proj = nn.Linear(d_model, ffn_dim, bias=False)\n        self.up_proj = nn.Linear(d_model, ffn_dim, bias=False)\n        self.down_proj = nn.Linear(ffn_dim, d_model, bias=False)\n\n    def forward(self, x):\n        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))\n\nclass EncoderBlock(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        d = cfg["d_model"]\n        self.norm1 = RMSNorm(d, cfg["norm_eps"])\n        self.self_attn = GQAAttention(d, cfg["encoder_heads"], cfg["encoder_heads"], cfg["head_dim"], cfg["context_length"], cfg["rope_theta"], causal=False, use_rope=True)\n        self.norm2 = RMSNorm(d, cfg["norm_eps"])\n        self.ffn = SwiGLU(d, cfg["ffn_dim"])\n\n    def forward(self, x, attention_mask):\n        x = x + self.self_attn(self.norm1(x), attn_mask=attention_mask)[0]\n        x = x + self.ffn(self.norm2(x))\n        return x\n\nclass DecoderBlock(nn.Module):\n    def __init__(self, cfg, fusion=False):\n        super().__init__()\n        d = cfg["d_model"]\n        self.fusion = bool(fusion)\n        self.norm1 = RMSNorm(d, cfg["norm_eps"])\n        self.self_attn = GQAAttention(d, cfg["decoder_q_heads"], cfg["decoder_kv_heads"], cfg["head_dim"], cfg["context_length"], cfg["rope_theta"], causal=True, use_rope=True)\n        if self.fusion:\n            self.cross_norm = RMSNorm(d, cfg["norm_eps"])\n            self.case_cross_attn = GQAAttention(d, cfg["decoder_q_heads"], cfg["decoder_kv_heads"], cfg["head_dim"], cfg["context_length"], cfg["rope_theta"], causal=False, use_rope=False)\n            self.evidence_cross_attn = GQAAttention(d, cfg["decoder_q_heads"], cfg["decoder_kv_heads"], cfg["head_dim"], cfg["context_length"], cfg["rope_theta"], causal=False, use_rope=False)\n            gh = cfg["gate_hidden_dim"]\n            self.gate = nn.Sequential(nn.Linear(d, gh, bias=True), nn.Linear(gh, 1, bias=True))\n        self.norm2 = RMSNorm(d, cfg["norm_eps"])\n        self.ffn = SwiGLU(d, cfg["ffn_dim"])\n\n    def forward_full(self, x, evidence, case, evidence_mask, decoder_mask):\n        x = x + self.self_attn(self.norm1(x), attn_mask=decoder_mask)[0]\n        if self.fusion:\n            u = self.cross_norm(x)\n            case_out = self.case_cross_attn(u, case)[0]\n            evidence_out = self.evidence_cross_attn(u, evidence, attn_mask=evidence_mask)[0]\n            gate = torch.sigmoid(self.gate(u))\n            x = x + gate * evidence_out + (1.0 - gate) * case_out\n        x = x + self.ffn(self.norm2(x))\n        return x\n\n    def forward_step(self, x, evidence, case, evidence_mask, cache, position):\n        y, new_cache = self.self_attn(self.norm1(x), cache=cache, use_cache=True, position_offset=position)\n        x = x + y\n        if self.fusion:\n            u = self.cross_norm(x)\n            case_out = self.case_cross_attn(u, case)[0]\n            evidence_out = self.evidence_cross_attn(u, evidence, attn_mask=evidence_mask)[0]\n            gate = torch.sigmoid(self.gate(u))\n            x = x + gate * evidence_out + (1.0 - gate) * case_out\n        x = x + self.ffn(self.norm2(x))\n        return x, new_cache\n\nclass DMLC150(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        self.cfg = dict(cfg)\n        d = int(cfg["d_model"])\n        V = int(cfg["vocab_size"])\n        self.token_embedding = nn.Embedding(V, d)\n        self.encoder = nn.ModuleList([EncoderBlock(cfg) for _ in range(cfg["encoder_layers"])])\n        self.encoder_final_norm = RMSNorm(d, cfg["norm_eps"])\n        self.case_memory = nn.Parameter(torch.empty(cfg["case_memory_slots"], d))\n        self.case_memory_slots = int(cfg["case_memory_slots"])\n        self.case_memory_attention = GQAAttention(d, cfg["decoder_q_heads"], cfg["decoder_kv_heads"], cfg["head_dim"], cfg["context_length"], cfg["rope_theta"], causal=False, use_rope=False)\n        self.case_memory_norm = RMSNorm(d, cfg["norm_eps"])\n        fusion = set(int(x) for x in cfg["memory_fusion_layers"])\n        self.decoder = nn.ModuleList([DecoderBlock(cfg, i in fusion) for i in range(1, cfg["decoder_layers"] + 1)])\n        self.fusion_layers = nn.ModuleDict({str(i): self.decoder[i - 1] for i in sorted(fusion)})\n        self.decoder_final_norm = RMSNorm(d, cfg["norm_eps"])\n        self.decoder_start = nn.Parameter(torch.empty(d))\n        self.lm_head = nn.Linear(d, V, bias=False)\n        self.lm_head.weight = self.token_embedding.weight\n        self._reset_parameters(cfg)\n\n    def _reset_parameters(self, cfg):\n        total_residual_layers = cfg["encoder_layers"] + cfg["decoder_layers"]\n        small_std = 0.02 / math.sqrt(max(1.0, 2.0 * total_residual_layers))\n        for name, module in self.named_modules():\n            if isinstance(module, nn.Linear):\n                use_small = name.endswith("o_proj") or name.endswith("down_proj")\n                std = small_std if use_small else 0.02\n                nn.init.normal_(module.weight, mean=0.0, std=std)\n                if module.bias is not None:\n                    nn.init.zeros_(module.bias)\n            elif isinstance(module, nn.Embedding):\n                nn.init.normal_(module.weight, mean=0.0, std=0.02)\n            elif isinstance(module, RMSNorm):\n                nn.init.ones_(module.weight)\n        nn.init.normal_(self.case_memory, mean=0.0, std=0.02)\n        nn.init.normal_(self.decoder_start, mean=0.0, std=0.02)\n        # Start with an unbiased 50/50 evidence-vs-case mixture.\n        for layer in self.fusion_layers.values():\n            nn.init.zeros_(layer.gate[-1].weight)\n            nn.init.zeros_(layer.gate[-1].bias)\n\n    def _encode(self, src_ids, src_attention_mask):\n        x = self.token_embedding(src_ids)\n        for layer in self.encoder:\n            if self.training and self.cfg.get("gradient_checkpointing", True):\n                x = checkpoint(layer, x, src_attention_mask, use_reentrant=False)\n            else:\n                x = layer(x, src_attention_mask)\n        evidence = self.encoder_final_norm(x)\n        case_seed = self.case_memory.unsqueeze(0).expand(src_ids.size(0), -1, -1)\n        case = case_seed + self.case_memory_attention(case_seed, evidence, attn_mask=src_attention_mask)[0]\n        case = self.case_memory_norm(case)\n        return evidence, case\n\n    def encode_prompt(self, src_ids, src_attention_mask=None):\n        if src_ids.dim() != 2:\n            raise ValueError("src_ids must be [B,S].")\n        if src_ids.size(1) > self.cfg["context_length"]:\n            raise RuntimeError("Source sequence exceeds context length.")\n        evidence, case = self._encode(src_ids, src_attention_mask)\n        return {"evidence_memory": evidence, "case_memory": case, "src_attention_mask": src_attention_mask}\n\n    def decode_teacher_forced(self, decoder_input_ids, memory, decoder_attention_mask=None):\n        if decoder_input_ids.dim() != 2:\n            raise ValueError("decoder_input_ids must be [B,T].")\n        B, T = decoder_input_ids.shape\n        if T > self.cfg["context_length"]:\n            raise RuntimeError("Target sequence exceeds decoder context length.")\n        start = self.decoder_start.view(1, 1, -1).expand(B, 1, -1)\n        shifted = self.token_embedding(decoder_input_ids[:, :-1]) if T > 1 else decoder_input_ids.new_empty((B, 0, self.cfg["d_model"]), dtype=torch.float32)\n        x = torch.cat([start, shifted], dim=1)\n        T2 = x.size(1)\n        if decoder_attention_mask is None:\n            causal = None  # Let SDPA use its fused causal path.\n        else:\n            causal = torch.tril(torch.ones((T2, T2), dtype=torch.bool, device=x.device))[None, None]\n            causal = causal & decoder_attention_mask[:, None, None, :]\n        evidence = memory["evidence_memory"]\n        case = memory["case_memory"]\n        evidence_mask = memory["src_attention_mask"]\n        for i, layer in enumerate(self.decoder, start=1):\n            if self.training and self.cfg.get("gradient_checkpointing", True):\n                x = checkpoint(layer.forward_full, x, evidence, case, evidence_mask, causal, use_reentrant=False)\n            else:\n                x = layer.forward_full(x, evidence, case, evidence_mask, causal)\n        return self.decoder_final_norm(x)\n\n    def forward(self, src_ids, decoder_input_ids=None, labels=None, src_attention_mask=None, decoder_attention_mask=None, return_logits=True):\n        if labels is not None and decoder_input_ids is None:\n            decoder_input_ids = labels\n        if decoder_input_ids is None:\n            raise ValueError("Provide decoder_input_ids or labels.")\n        memory = self.encode_prompt(src_ids, src_attention_mask)\n        hidden = self.decode_teacher_forced(decoder_input_ids, memory, decoder_attention_mask)\n        logits = F.linear(hidden, self.token_embedding.weight)\n        out = {}\n        if labels is not None:\n            out["loss"] = F.cross_entropy(logits.reshape(-1, logits.size(-1)), labels.reshape(-1), ignore_index=-100)\n        if return_logits:\n            out["logits"] = logits\n        return out\n\n    @torch.no_grad()\n    def generate(self, src_ids, src_attention_mask=None, max_new_tokens=512, temperature=0.10, top_p=0.95, top_k=20, special_token_ids=None):\n        if special_token_ids is None:\n            special_token_ids = set()\n        memory = self.encode_prompt(src_ids, src_attention_mask)\n        B = src_ids.size(0)\n        x = self.decoder_start.view(1, 1, -1).expand(B, 1, -1)\n        caches = [None] * len(self.decoder)\n        generated = []\n        for position in range(int(max_new_tokens)):\n            for i, layer in enumerate(self.decoder):\n                x, caches[i] = layer.forward_step(x, memory["evidence_memory"], memory["case_memory"], memory["src_attention_mask"], caches[i], position)\n            logits = F.linear(self.decoder_final_norm(x)[:, -1], self.token_embedding.weight).float()\n            for token_id in special_token_ids:\n                if 0 <= int(token_id) < logits.size(-1):\n                    logits[:, int(token_id)] = -float("inf")\n            if temperature is None or float(temperature) <= 0:\n                next_ids = torch.argmax(logits, dim=-1)\n            else:\n                logits = logits / max(float(temperature), 1e-5)\n                if top_k and 0 < int(top_k) < logits.size(-1):\n                    values, _ = torch.topk(logits, int(top_k), dim=-1)\n                    cutoff = values[:, -1, None]\n                    logits = torch.where(logits < cutoff, torch.full_like(logits, -float("inf")), logits)\n                if top_p is not None and 0 < float(top_p) < 1:\n                    sorted_logits, sorted_indices = torch.sort(logits, descending=True, dim=-1)\n                    probs = torch.softmax(sorted_logits, dim=-1)\n                    cumulative = torch.cumsum(probs, dim=-1)\n                    remove = cumulative > float(top_p)\n                    remove[:, 1:] = remove[:, :-1].clone()\n                    remove[:, 0] = False\n                    sorted_logits = sorted_logits.masked_fill(remove, -float("inf"))\n                    logits = torch.full_like(logits, -float("inf")).scatter(1, sorted_indices, sorted_logits)\n                next_ids = torch.multinomial(torch.softmax(logits, dim=-1), 1).squeeze(-1)\n            generated.append(next_ids)\n            x = self.token_embedding(next_ids).unsqueeze(1)\n        return torch.stack(generated, dim=1) if generated else src_ids.new_empty((B, 0))'
ARCH_SOURCE['D'] = 'class RMSNorm(nn.Module):\n    def __init__(self, dim, eps=1e-5):\n        super().__init__()\n        self.weight = nn.Parameter(torch.ones(dim))\n        self.eps = eps\n\n    def forward(self, x):\n        rms = torch.rsqrt(\n            x.float().pow(2).mean(dim=-1, keepdim=True) + self.eps\n        ).to(x.dtype)\n        return x * rms * self.weight.to(x.dtype)\n\nclass RoPE(nn.Module):\n    def __init__(self, head_dim, max_len, theta=10000.0):\n        super().__init__()\n        inv_freq = 1.0 / (\n            theta ** (\n                torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim\n            )\n        )\n        pos = torch.arange(max_len, dtype=torch.float32)\n        freqs = torch.outer(pos, inv_freq)\n        self.register_buffer("cos", freqs.cos(), persistent=False)\n        self.register_buffer("sin", freqs.sin(), persistent=False)\n\n    def rotate(self, x, offset=0):\n        t = x.size(-2)\n        c = self.cos[offset:offset+t].to(x.device, x.dtype)[None, None, :, :]\n        s = self.sin[offset:offset+t].to(x.device, x.dtype)[None, None, :, :]\n        xe, xo = x[..., 0::2], x[..., 1::2]\n        y = torch.empty_like(x)\n        y[..., 0::2] = xe * c - xo * s\n        y[..., 1::2] = xe * s + xo * c\n        return y\n\nclass PrefixGQAAttention(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        d = cfg["d_model"]\n        H = cfg["n_heads"]\n        KV = cfg["n_kv_heads"]\n        hd = cfg["head_dim"]\n\n        self.H = H\n        self.KV = KV\n        self.hd = hd\n        self.group = H // KV\n\n        self.q_proj = nn.Linear(d, H * hd, bias=False)\n        self.k_proj = nn.Linear(d, KV * hd, bias=False)\n        self.v_proj = nn.Linear(d, KV * hd, bias=False)\n        self.o_proj = nn.Linear(d, d, bias=False)\n\n        self.rope = RoPE(hd, cfg["context_length"], cfg["rope_theta"])\n        self.q_norm = RMSNorm(hd, cfg["norm_eps"]) if cfg.get("qk_norm") else nn.Identity()\n        self.k_norm = RMSNorm(hd, cfg["norm_eps"]) if cfg.get("qk_norm") else nn.Identity()\n\n        suffix_len = cfg["context_length"] - cfg["prefix_length"]\n        prefix_len = cfg["prefix_length"]\n\n        # Small suffix mask: [suffix_queries, prefix+suffix_keys].\n        mask = torch.ones(\n            suffix_len, prefix_len + suffix_len, dtype=torch.bool\n        )\n        mask[:, prefix_len:] = torch.tril(\n            torch.ones(suffix_len, suffix_len, dtype=torch.bool)\n        )\n        self.register_buffer(\n            "suffix_mask",\n            mask[None, None, :, :],\n            persistent=False,\n        )\n\n    def forward(self, x, prefix_len, past_kv=None, use_cache=False):\n        B, T, D = x.shape\n        past_len = 0 if past_kv is None else past_kv[0].size(2)\n\n        q = self.q_proj(x).view(B, T, self.H, self.hd).transpose(1, 2)\n        k = self.k_proj(x).view(B, T, self.KV, self.hd).transpose(1, 2)\n        v = self.v_proj(x).view(B, T, self.KV, self.hd).transpose(1, 2)\n\n        q = self.q_norm(self.rope.rotate(q, past_len))\n        k = self.k_norm(self.rope.rotate(k, past_len))\n\n        if past_kv is not None:\n            pk, pv = past_kv\n            k = torch.cat([pk, k], dim=2)\n            v = torch.cat([pv, v], dim=2)\n\n        ka = k.repeat_interleave(self.group, dim=1)\n        va = v.repeat_interleave(self.group, dim=1)\n\n        if past_kv is not None:\n            y = F.scaled_dot_product_attention(\n                q, ka, va,\n                attn_mask=None,\n                dropout_p=0.0,\n                is_causal=False,\n            )\n        else:\n            p = int(prefix_len)\n            q_pref, q_suf = q[:, :, :p], q[:, :, p:]\n            k_all, v_all = ka, va\n\n            y_pref = F.scaled_dot_product_attention(\n                q_pref, k_all[:, :, :p], v_all[:, :, :p],\n                attn_mask=None,\n                dropout_p=0.0,\n                is_causal=False,\n            )\n\n            mask = self.suffix_mask[:, :, :q_suf.size(2), :k_all.size(2)]\n            y_suf = F.scaled_dot_product_attention(\n                q_suf, k_all, v_all,\n                attn_mask=mask,\n                dropout_p=0.0,\n                is_causal=False,\n            )\n\n            y = torch.cat([y_pref, y_suf], dim=2)\n\n        y = y.transpose(1, 2).contiguous().view(B, T, D)\n        out = self.o_proj(y)\n\n        return (out, (k, v)) if use_cache else out\n\nclass SwiGLU(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        d, f = cfg["d_model"], cfg["ffn_dim"]\n        self.gate = nn.Linear(d, f, bias=False)\n        self.up = nn.Linear(d, f, bias=False)\n        self.down = nn.Linear(f, d, bias=False)\n\n    def forward(self, x):\n        return self.down(F.silu(self.gate(x)) * self.up(x))\n\nclass PrefixBlock(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        self.norm1 = RMSNorm(cfg["d_model"], cfg["norm_eps"])\n        self.attn = PrefixGQAAttention(cfg)\n        self.norm2 = RMSNorm(cfg["d_model"], cfg["norm_eps"])\n        self.ffn = SwiGLU(cfg)\n\n    def forward(self, x, prefix_len, past_kv=None, use_cache=False):\n        if use_cache:\n            a, present = self.attn(\n                self.norm1(x),\n                prefix_len,\n                past_kv=past_kv,\n                use_cache=True,\n            )\n            x = x + a\n            x = x + self.ffn(self.norm2(x))\n            return x, present\n\n        x = x + self.attn(self.norm1(x), prefix_len)\n        x = x + self.ffn(self.norm2(x))\n        return x\n\nclass LegalPrefixFormer145(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        self.cfg = dict(cfg)\n        d = cfg["d_model"]\n\n        self.token_embedding = nn.Embedding(cfg["vocab_size"], d)\n        self.blocks = nn.ModuleList([\n            PrefixBlock(cfg) for _ in range(cfg["n_layers"])\n        ])\n        self.final_norm = RMSNorm(d, cfg["norm_eps"])\n        self.lm_head = nn.Linear(d, cfg["vocab_size"], bias=False)\n\n        if cfg["tie_embeddings"]:\n            self.lm_head.weight = self.token_embedding.weight\n\n        self.apply(self._init_weights)\n\n        residual_scale = 1.0 / math.sqrt(2.0 * cfg["n_layers"])\n        for block in self.blocks:\n            nn.init.normal_(\n                block.attn.o_proj.weight,\n                mean=0.0,\n                std=0.02 * residual_scale,\n            )\n            nn.init.normal_(\n                block.ffn.down.weight,\n                mean=0.0,\n                std=0.02 * residual_scale,\n            )\n\n    def _init_weights(self, module):\n        if isinstance(module, nn.Linear):\n            nn.init.normal_(module.weight, mean=0.0, std=0.02)\n            if module.bias is not None:\n                nn.init.zeros_(module.bias)\n        elif isinstance(module, nn.Embedding):\n            nn.init.normal_(module.weight, mean=0.0, std=0.02)\n\n    def _run_block(self, block, x, prefix_len):\n        return block(x, prefix_len)\n\n    def forward(\n        self,\n        input_ids,\n        labels=None,\n        prefix_len=None,\n        past_key_values=None,\n        use_cache=False,\n        return_logits=None,\n    ):\n        if prefix_len is None:\n            prefix_len = self.cfg["prefix_length"]\n\n        x = self.token_embedding(input_ids)\n        presents = []\n\n        for i, block in enumerate(self.blocks):\n            pkv = None if past_key_values is None else past_key_values[i]\n\n            if use_cache:\n                x, present = block(\n                    x,\n                    prefix_len,\n                    past_kv=pkv,\n                    use_cache=True,\n                )\n                presents.append(present)\n            elif self.training and self.cfg.get("activation_checkpointing", False):\n                x = activation_checkpoint(\n                    self._run_block,\n                    block,\n                    x,\n                    prefix_len,\n                    use_reentrant=False,\n                )\n            else:\n                x = block(x, prefix_len)\n\n        x = self.final_norm(x)\n\n        loss = None\n        logits = None\n\n        if labels is not None:\n            # Input layout:\n            # prefix[0:1024], BOS[1024], target-input[1025:1280].\n            # Prediction logits at positions [1024:1280] map to 256 labels.\n            target_hidden = x[:, prefix_len:, :]\n            target_logits = self.lm_head(target_hidden)\n            target_labels = labels[:, prefix_len:]\n            loss = F.cross_entropy(\n                target_logits.reshape(-1, target_logits.size(-1)).float(),\n                target_labels.reshape(-1),\n                ignore_index=-100,\n            )\n\n        if return_logits is None:\n            return_logits = labels is None or use_cache\n\n        if return_logits:\n            logits = self.lm_head(x)\n\n        return {\n            "logits": logits,\n            "loss": loss,\n            "past_key_values": tuple(presents) if use_cache else None,\n        }'
ARCH_NS = {}
for _arch, _source in ARCH_SOURCE.items():
    _ns = {"torch":torch, "nn":nn, "F":F, "math":math, "np":np, "checkpoint":checkpoint, "activation_checkpoint":checkpoint}
    exec(compile(_source, f'<phase1_arch_{_arch}>', 'exec'), _ns, _ns)
    if ARCH_CLASS_NAMES[_arch] not in _ns: raise RuntimeError(f'Missing class for architecture {_arch}')
    ARCH_NS[_arch] = _ns
    print(f'Architecture {_arch}: source classes loaded | {ARCH_CLASS_NAMES[_arch]}')
print('All four existing Phase-1 architecture implementations are ready.')

In [ ]:

def set_child(root, dotted, child):
    parts = dotted.split("."); parent = root
    for q in parts[:-1]: parent = getattr(parent, q)
    setattr(parent, parts[-1], child)

class LowRankLinear(nn.Module):
    def __init__(self, base, rank=8, alpha=16.0, lorma=False):
        super().__init__()
        self.base = base
        for p in self.base.parameters(): p.requires_grad = False
        self.rank, self.alpha, self.lorma = int(rank), float(alpha), bool(lorma)
        if lorma:
            outdim = base.out_features
            self.A = nn.Parameter(torch.empty(self.rank, outdim))
            self.B = nn.Parameter(torch.zeros(outdim, self.rank))
            nn.init.normal_(self.A, std=0.01)
        else:
            self.A = nn.Parameter(torch.empty(self.rank, base.in_features))
            self.B = nn.Parameter(torch.zeros(base.out_features, self.rank))
            nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
    def forward(self, x):
        y = self.base(x)
        if self.lorma:
            delta = F.linear(F.linear(y, self.A), self.B)
        else:
            delta = F.linear(F.linear(x, self.A), self.B)
        return y + delta * (self.alpha / self.rank)

class BottleneckAdapter(nn.Module):
    def __init__(self, dim, bottleneck=48):
        super().__init__()
        self.down = nn.Linear(dim, bottleneck)
        self.act = nn.GELU()
        self.up = nn.Linear(bottleneck, dim)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)
    def forward(self, x):
        return x + self.up(self.act(self.down(x)))

def install_low_rank(model, lorma=False, rank=16):
    targets = []
    for name, mod in list(model.named_modules()):
        if not name or not isinstance(mod, nn.Linear): continue
        low = name.lower()
        if any(k in low for k in ("attn.","attention.","ffn.","gate.")) and not low.endswith("lm_head"):
            targets.append((name, mod))
    if not targets: raise RuntimeError("No supported linear targets for low-rank adaptation.")
    for name, mod in targets: set_child(model, name, LowRankLinear(mod, rank=rank, alpha=2*rank, lorma=lorma))
    return len(targets)

def install_adapters(model, arch, style="houlsby", bottleneck=48):
    groups = []
    if hasattr(model, "blocks"): groups.append(list(model.blocks))
    if arch == "C":
        if hasattr(model, "encoder"): groups.extend([list(model.encoder), list(model.decoder)])
    if not groups: raise RuntimeError("No Transformer layers available for adapter installation.")
    selected = []
    for group in groups:
        selected.extend(group[-min(4,len(group)):])
    adapters = nn.ModuleDict()
    handles = []
    n = 0
    for layer in selected:
        modules = []
        if style == "houlsby":
            for sub in ("attn","ffn"):
                m = getattr(layer, sub, None)
                if isinstance(m, nn.Module):
                    key = str(n); adapters[key] = BottleneckAdapter(int(model.cfg["d_model"]), bottleneck)
                    def hook_factory(k):
                        def hook(module, inputs, output):
                            adapter = model.phase6_adapters[k]
                            if isinstance(output, tuple): return (adapter(output[0]),) + output[1:]
                            return adapter(output)
                        return hook
                    handles.append(m.register_forward_hook(hook_factory(key))); n += 1
        else:
            key = str(n); adapters[key] = BottleneckAdapter(int(model.cfg["d_model"]), bottleneck)
            def hook_factory(k):
                def hook(module, inputs, output):
                    adapter = model.phase6_adapters[k]
                    if isinstance(output, tuple): return (adapter(output[0]),) + output[1:]
                    return adapter(output)
                return hook
            handles.append(layer.register_forward_hook(hook_factory(key))); n += 1
    model.add_module("phase6_adapters", adapters)
    model._phase6_hook_handles = handles
    return n

def install_soft_prefix(model, length=12):
    if not hasattr(model, "token_embedding"): raise RuntimeError("Soft prefix requires token_embedding.")
    dim = int(model.cfg["d_model"])
    model.phase6_soft_prefix = nn.Parameter(torch.empty(length, dim))
    nn.init.normal_(model.phase6_soft_prefix, mean=0.0, std=0.02)
    original_forward = model.token_embedding.forward
    def prefixed(ids):
        em = original_forward(ids)
        # Initial full prompt gets the learned virtual prefix. During cached
        # autoregressive decoding, single-token calls must NOT prepend it again.
        if ids.size(1) == 1:
            return em
        p = model.phase6_soft_prefix.unsqueeze(0).expand(ids.size(0), -1, -1).to(em.dtype)
        return torch.cat([p, em], dim=1)
    model.token_embedding.forward = prefixed
    model._phase6_original_embedding_forward = original_forward
    return length

def load_phase1_model(arch, method):
    if method == "kd": method = "lora"  # KD student uses the same supported LoRA student adapter.
    if arch not in CHECKPOINTS: raise FileNotFoundError(f"Architecture {arch} checkpoint is not available.")
    p = Path(CHECKPOINTS[arch])
    try: payload = torch.load(p, map_location="cpu", weights_only=False, mmap=True)
    except Exception: payload = torch.load(p, map_location="cpu", weights_only=False)
    cfg = get_cfg(payload); state = get_state(payload)
    if not cfg or state is None: raise RuntimeError(f"Checkpoint {p} has no model config/state.")
    cls = ARCH_NS[arch][ARCH_CLASS_NAMES[arch]]
    model = cls(cfg)
    if state and all(str(k).startswith("module.") for k in state):
        state = {str(k)[7:]:v for k,v in state.items()}
    target = model.state_dict()
    if arch == "C":
        # Preserve the existing, checkpoint-tested DMLC gate key compatibility mapping.
        remapped = {}
        for k,v in state.items():
            nk = re.sub(r"(\.gate)\.2(?=\.(?:weight|bias)$)", r"\1.1", k)
            if nk not in target or tuple(target[nk].shape) != tuple(v.shape): nk = k
            remapped[nk] = v
        state = remapped
    missing, unexpected = model.load_state_dict(state, strict=False)
    allowed_missing = [k for k in missing if k.endswith("lm_head.weight") and cfg.get("tie_embeddings", False)]
    bad_missing = [k for k in missing if k not in allowed_missing]
    if bad_missing or unexpected:
        raise RuntimeError(f"Checkpoint mismatch {arch}: missing={bad_missing[:5]} unexpected={unexpected[:5]}")
    del payload, state
    for p0 in model.parameters(): p0.requires_grad = False
    if method == "full":
        for p0 in model.parameters(): p0.requires_grad = True
    elif method == "lora":
        install_low_rank(model, lorma=False, rank=16)
    elif method == "lorma":
        install_low_rank(model, lorma=True)
    elif method == "houlsby":
        install_adapters(model, arch, "houlsby", bottleneck=48)
    elif method == "pfeiffer":
        install_adapters(model, arch, "pfeiffer", bottleneck=32)
    elif method == "prefix":
        if arch == "C": raise RuntimeError("Soft prefix experiment uses decoder-only model B, not DMLC.")
        install_soft_prefix(model, length=12)
    else:
        raise ValueError("Unsupported Phase-6 adaptation method: "+method)
    # Keep the LM head and base decoder intact for generative fine-tuning.
    model = model.to(DEVICE)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Loaded Phase-1 {arch} | method={method} | trainable={trainable:,} | checkpoint={CHECKPOINTS[arch]}")
    return model, cfg

print("Phase-6 generative loader and PEFT installers ready.")


In [ ]:

def take_head_tail(ids, limit):
    if len(ids) <= limit: return ids
    left = int(limit * 0.75)
    right = limit - left
    return ids[:left] + (ids[-right:] if right else [])

def encode_records(records, labeled):
    encoded = []
    for i, row in enumerate(records):
        prompt = (
            "You are a legal summarization assistant. Write a concise, factual, non-redundant "
            "summary preserving the material facts, issues, legal reasoning, statutes considered, "
            "and final decision. Do not add facts absent from the document.\n\n"
            "LEGAL DOCUMENT:\n" + row["document"] + "\n\nLEGAL SUMMARY:\n"
        )
        src = take_head_tail(TOKENIZER.encode(prompt).ids, MAX_SOURCE_TOKENS)
        obj = {"id":row["id"], "src":src}
        if labeled:
            tgt = TOKENIZER.encode(row["summary"]).ids
            if EOS_ID is not None: tgt = tgt[:MAX_TARGET_TOKENS-1] + [EOS_ID]
            else: tgt = tgt[:MAX_TARGET_TOKENS]
            obj["tgt"] = tgt
        encoded.append(obj)
        if (i+1) % 1000 == 0: print(f"Tokenized {i+1:,}/{len(records):,}")
    return encoded

CACHE_SIG = hashlib.sha256((TOKENIZER_SHA256+str(len(TRAIN_RECORDS))+str(len(VAL_RECORDS))+str(len(TEST_RECORDS))).encode()).hexdigest()[:16]
cache_file = CACHE_DIR / f"phase6_tokens_{CACHE_SIG}.json"
if cache_file.exists():
    try:
        cached = json.loads(cache_file.read_text(encoding="utf-8"))
        TRAIN_ENC, VAL_ENC, TEST_ENC = cached["train"], cached["val"], cached["test"]
        print("Restored token cache:", cache_file)
    except Exception:
        TRAIN_ENC = encode_records(TRAIN_RECORDS, True)
        VAL_ENC = encode_records(VAL_RECORDS, True)
        TEST_ENC = encode_records(TEST_RECORDS, False)
else:
    TRAIN_ENC = encode_records(TRAIN_RECORDS, True)
    VAL_ENC = encode_records(VAL_RECORDS, True)
    TEST_ENC = encode_records(TEST_RECORDS, False)
    cache_file.write_text(json.dumps({"train":TRAIN_ENC,"val":VAL_ENC,"test":TEST_ENC}), encoding="utf-8")
print("Tokenized sets:",len(TRAIN_ENC),len(VAL_ENC),len(TEST_ENC))


In [ ]:

def make_training_tensors(arch, method, item):
    src = list(item["src"])
    tgt = list(item["tgt"])
    if not tgt: tgt = [EOS_ID if EOS_ID is not None else PAD_ID]
    if arch == "C":
        # DMLC-150 shifts decoder_input_ids internally using decoder_start.
        return {
            "src_ids": torch.tensor([src], dtype=torch.long, device=DEVICE),
            "src_mask": torch.ones((1, len(src)), dtype=torch.bool, device=DEVICE),
            "dec_ids": torch.tensor([tgt], dtype=torch.long, device=DEVICE),
            "labels": torch.tensor([tgt], dtype=torch.long, device=DEVICE),
        }
    inp = src + [BOS_ID] + tgt[:-1]
    labels = [-100] * len(src) + tgt
    if method == "prefix":
        labels = [-100] * 12 + labels
    return {
        "input_ids": torch.tensor([inp], dtype=torch.long, device=DEVICE),
        "labels": torch.tensor([labels], dtype=torch.long, device=DEVICE),
        "prefix_len": len(src),
    }

def model_loss(model, arch, method, item):
    batch = make_training_tensors(arch, method, item)
    if arch == "C":
        out = model(src_ids=batch["src_ids"], decoder_input_ids=batch["dec_ids"],
                    labels=batch["labels"], src_attention_mask=batch["src_mask"])
        return out["loss"]
    if arch == "D":
        out = model(input_ids=batch["input_ids"], labels=batch["labels"],
                    prefix_len=batch["prefix_len"], return_logits=False)
        return out["loss"]
    out = model(input_ids=batch["input_ids"], labels=batch["labels"])
    return out["loss"]

def atomic_torch_save(obj, path):
    path = Path(path); tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(obj, tmp); os.replace(tmp, path)

def validation_loss(model, arch, method, val_items, limit=None):
    model.eval(); values=[]
    items = val_items if limit is None else val_items[:limit]
    with torch.no_grad():
        for item in items:
            loss = model_loss(model, arch, method, item)
            if loss is not None and torch.isfinite(loss): values.append(float(loss.detach().float().cpu()))
    model.train()
    return float(np.mean(values)) if values else float("inf")

def generate_one(model, arch, method, item, max_new_tokens=GEN_MAX_NEW_TOKENS):
    """Greedy generation with explicit EOS stopping for every architecture."""
    model.eval()
    src = list(item["src"])
    gen = []
    with torch.inference_mode():
        if arch == "C":
            # DMLC150.generate in the Phase-1 source runs to max_new_tokens even
            # after EOS. Use its native cached decoder layers here and stop at EOS.
            ids = torch.tensor([src], dtype=torch.long, device=DEVICE)
            mask = torch.ones_like(ids, dtype=torch.bool)
            memory = model.encode_prompt(ids, mask)
            x = model.decoder_start.view(1, 1, -1).expand(ids.size(0), 1, -1)
            caches = [None] * len(model.decoder)
            for position in range(int(max_new_tokens)):
                for layer_index, layer in enumerate(model.decoder):
                    x, caches[layer_index] = layer.forward_step(
                        x, memory["evidence_memory"], memory["case_memory"],
                        memory["src_attention_mask"], caches[layer_index], position
                    )
                logits = F.linear(
                    model.decoder_final_norm(x)[:, -1], model.token_embedding.weight
                ).float()
                if PAD_ID is not None and PAD_ID != EOS_ID:
                    logits[:, PAD_ID] = -float("inf")
                if BOS_ID is not None and BOS_ID != EOS_ID:
                    logits[:, BOS_ID] = -float("inf")
                next_ids = torch.argmax(logits, dim=-1)
                next_id = int(next_ids.item())
                if EOS_ID is not None and next_id == EOS_ID:
                    break
                gen.append(next_id)
                x = model.token_embedding(next_ids).unsqueeze(1)
        else:
            # A/B/D use the Phase-1 decoder KV-cache interface.
            seq = torch.tensor([src + [BOS_ID]], dtype=torch.long, device=DEVICE)
            past = None
            context_limit = int(model.cfg.get("context_length", 2048))
            soft_prefix = 12 if method == "prefix" else 0
            for step in range(int(max_new_tokens)):
                if len(src) + 1 + soft_prefix + step >= context_limit:
                    break
                if arch == "D":
                    out = model(input_ids=seq, prefix_len=len(src), past_key_values=past,
                                use_cache=True, return_logits=True)
                else:
                    out = model(input_ids=seq, past_key_values=past, use_cache=True)
                logits = out["logits"][:, -1, :].float()
                if PAD_ID is not None and PAD_ID != EOS_ID:
                    logits[:, PAD_ID] = -float("inf")
                if BOS_ID is not None and BOS_ID != EOS_ID:
                    logits[:, BOS_ID] = -float("inf")
                nxt = int(torch.argmax(logits, dim=-1).item())
                if EOS_ID is not None and nxt == EOS_ID:
                    break
                gen.append(nxt)
                past = out.get("past_key_values")
                seq = torch.tensor([[nxt]], dtype=torch.long, device=DEVICE)
                if past is None:
                    seq = torch.tensor([src + [BOS_ID] + gen], dtype=torch.long, device=DEVICE)

    # Tokenizer's special-token filter removes any registered control tokens.
    text = TOKENIZER.decode(gen, skip_special_tokens=True).strip()
    text = re.sub(r"\s+", " ", text)
    return text

def make_submission(predictions, exp_name):
    out_dir = SUB_DIR / exp_name
    out_dir.mkdir(parents=True, exist_ok=True)
    pred_json = out_dir / "prediction.json"
    pred_json.write_text(json.dumps(predictions, ensure_ascii=False, indent=2), encoding="utf-8")
    zip_path = out_dir / "submission.zip"
    tmp = zip_path.with_suffix(".tmp.zip")
    with zipfile.ZipFile(tmp, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        zf.write(pred_json, arcname="prediction.json")
    with zipfile.ZipFile(tmp) as zf:
        assert zf.testzip() is None and zf.namelist() == ["prediction.json"]
        packaged = json.loads(zf.read("prediction.json").decode("utf-8"))
    assert packaged == predictions
    os.replace(tmp, zip_path)
    return zip_path

def generate_test_submission(model, arch, method, exp_name):
    state_path = SUB_DIR / exp_name / "generation_state.json"
    state_path.parent.mkdir(parents=True, exist_ok=True)
    saved = {}
    if state_path.exists():
        try:
            prior = json.loads(state_path.read_text(encoding="utf-8"))
            saved = prior.get("predictions", {})
        except Exception:
            saved = {}
    expected_ids = [str(r["id"]) for r in TEST_ENC]
    # Drop stale/foreign IDs from an earlier data version.
    saved = {str(k): str(v) for k, v in saved.items() if str(k) in set(expected_ids)}
    for i, item in enumerate(TEST_ENC):
        rid = str(item["id"])
        if rid in saved and saved[rid].strip():
            continue
        summary = generate_one(model, arch, method, item)
        saved[rid] = summary
        if (i + 1) % 10 == 0 or i + 1 == len(TEST_ENC):
            tmp_state = state_path.with_suffix(".tmp")
            tmp_state.write_text(json.dumps({"predictions": saved, "next_index": i + 1}, ensure_ascii=False), encoding="utf-8")
            os.replace(tmp_state, state_path)
            print(f"{exp_name}: generated {len(saved):,}/{len(TEST_ENC):,}")
    predictions = [{"id": r["id"], "summary": saved[str(r["id"])]} for r in TEST_ENC]
    assert len(predictions) == len(TEST_RECORDS)
    assert [str(x["id"]) for x in predictions] == [str(x["id"]) for x in TEST_RECORDS]
    if any(not isinstance(x["summary"], str) or not x["summary"].strip() for x in predictions):
        raise RuntimeError(f"{exp_name}: one or more generated summaries are empty; ZIP not created.")
    return make_submission(predictions, exp_name)

def save_run_result(name, payload):
    path = EXP_DIR / name / "result.json"
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")

print("Trainer, resumable checkpoints, cached generation, and submission ZIP helpers ready.")

In [ ]:

if not torch.cuda.is_available():
    raise RuntimeError("A Kaggle GPU is required for this decoding experiment. Select a GPU accelerator and rerun.")
if "C" not in CHECKPOINTS:
    raise FileNotFoundError("Architecture-C / DMLC-150 Phase-1 checkpoint was not detected. Attach the same Phase-1 C dataset used by E7.")

EXPERIMENT_NAME = "E8_C_LoRA_R16_RepetitionControlled"
E8_DIR = EXP_DIR / EXPERIMENT_NAME
E8_DIR.mkdir(parents=True, exist_ok=True)
E8_START_TIME = time.time()

# Reuse E7's saved adapter from this live /kaggle/working session or an attached
# Kaggle Notebook Output mounted under /kaggle/input. The E7 experiment directory
# name is preferred so another experiment's adapter is never selected accidentally.
expected_adapter = ROOT / "experiments" / "E7_C_LoRA_R16_FullData" / "best_adapter.pt"
search_roots = [Path("/kaggle/working"), Path("/kaggle/input")]
adapter_candidates = [expected_adapter]
for search_root in search_roots:
    if search_root.exists():
        for p in iter_files(search_root, suffixes={".pt", ".pth", ".ckpt"}):
            if p.name.lower() == "best_adapter.pt":
                adapter_candidates.append(p)

unique_candidates=[]
seen=set()
for p in adapter_candidates:
    p=Path(p)
    if p.is_file() and str(p) not in seen:
        seen.add(str(p)); unique_candidates.append(p)
# Exact experiment name is the strongest signal; expected working path wins ties.
unique_candidates.sort(key=lambda p: (
    0 if "E7_C_LoRA_R16_FullData" in str(p) else 1,
    0 if str(p) == str(expected_adapter) else 1,
    len(str(p))
))

if not unique_candidates:
    print("Searched roots:", [str(p) for p in search_roots])
    raise FileNotFoundError(
        "E7 best_adapter.pt was not found. If continuing in the original E7 notebook, "
        "run E8 cells in that same live kernel without restarting it. If this is a "
        "separate notebook, attach the E7 notebook output using Kaggle Add Input / "
        "Notebook Output, then rerun from Cell 1. Expected file: " + str(expected_adapter)
    )

ADAPTER_PATH = unique_candidates[0]
print("Using E7 rank-16 LoRA adapter:", ADAPTER_PATH)
print("Adapter file size: %.2f MiB" % (ADAPTER_PATH.stat().st_size / (1024**2)))
if len(unique_candidates) > 1:
    print("Other matching best_adapter.pt files found (not selected):", [str(x) for x in unique_candidates[1:5]])

# Rebuild the exact Phase-1 C + rank-16 LoRA parameter layout, then load E7's saved trainable tensors.
model, model_cfg = load_phase1_model("C", "lora")
trainable_params = [p for p in model.parameters() if p.requires_grad]
trainable_count = sum(p.numel() for p in trainable_params)
if trainable_count < 4_000_000:
    raise RuntimeError(f"Expected rank-16 LoRA adapter capacity; found only {trainable_count:,} trainable parameters.")

adapter_payload = torch.load(ADAPTER_PATH, map_location="cpu", weights_only=False)
if not isinstance(adapter_payload, dict) or "trainable_state" not in adapter_payload:
    raise RuntimeError("The selected file is not an E7 best_adapter.pt checkpoint: missing 'trainable_state'.")
if adapter_payload.get("arch") not in (None, "C"):
    raise RuntimeError(f"Adapter architecture mismatch: {adapter_payload.get('arch')!r}; expected 'C'.")
if adapter_payload.get("method") not in (None, "lora_rank16"):
    raise RuntimeError(f"Adapter method mismatch: {adapter_payload.get('method')!r}; expected 'lora_rank16'.")

named_params = dict(model.named_parameters())
loaded_keys=0
unknown=[]
for name, value in adapter_payload["trainable_state"].items():
    if name not in named_params or not named_params[name].requires_grad:
        unknown.append(name); continue
    named_params[name].data.copy_(value.to(device=named_params[name].device, dtype=named_params[name].dtype))
    loaded_keys += 1
if unknown:
    raise RuntimeError(f"E7 adapter contains incompatible parameter keys: {unknown[:8]}")
if loaded_keys < 100:
    raise RuntimeError(f"Only {loaded_keys} LoRA tensors loaded; adapter appears incomplete or incompatible.")
if int(adapter_payload.get("trainable_count", trainable_count)) != trainable_count:
    print("WARNING: saved trainable_count differs from rebuilt model:", adapter_payload.get("trainable_count"), trainable_count)

model.eval()
print(f"PASS | E7 LoRA loaded | trainable={trainable_count:,} | tensors={loaded_keys} | E7 step={adapter_payload.get('step', 'unknown')} | val_loss={adapter_payload.get('best_val_loss', 'unknown')}")
del adapter_payload
import gc
gc.collect(); torch.cuda.empty_cache()

In [ ]:

from collections import Counter
import subprocess, sys

# Lightweight ROUGE-1 / ROUGE-L implementations (no extra package required).
def _e8_words(text):
    return re.findall(r"[a-z0-9]+", str(text).lower())

def _e8_f1(overlap, pred_n, ref_n):
    if overlap <= 0 or pred_n <= 0 or ref_n <= 0: return 0.0
    p, r = overlap / pred_n, overlap / ref_n
    return 2*p*r/(p+r)

def e8_rouge1(pred, ref):
    p, r = _e8_words(pred), _e8_words(ref)
    return _e8_f1(sum((Counter(p) & Counter(r)).values()), len(p), len(r))

def e8_rouge_l(pred, ref):
    p, r = _e8_words(pred), _e8_words(ref)
    if not p or not r: return 0.0
    if len(r) > len(p): p, r = r, p
    prev=[0]*(len(r)+1)
    for tok in p:
        cur=[0]
        for j, rtok in enumerate(r, 1):
            cur.append(prev[j-1]+1 if tok==rtok else max(prev[j],cur[-1]))
        prev=cur
    return _e8_f1(prev[-1],len(p),len(r))

def e8_repetition_stats(text):
    sentences=[s.strip().lower() for s in re.split(r"(?<=[.!?])\s+", str(text)) if s.strip()]
    dup_sent=(len(sentences)-len(set(sentences)))/max(1,len(sentences))
    toks=_e8_words(text)
    grams=[tuple(toks[i:i+4]) for i in range(max(0,len(toks)-3))]
    dup4=(len(grams)-len(set(grams)))/max(1,len(grams))
    return float(dup_sent), float(dup4)

# DMLC-150 autoregressive decoding. Repetition penalty is applied to previously
# generated tokens; no-repeat n-gram blocking bans a token that would recreate
# an earlier n-gram. EOS is never n-gram-blocked.
def generate_c_controlled(model, item, max_new_tokens=160, repetition_penalty=1.0, no_repeat_ngram_size=0):
    model.eval()
    src=list(item["src"])
    gen=[]
    with torch.inference_mode():
        ids=torch.tensor([src],dtype=torch.long,device=DEVICE)
        mask=torch.ones_like(ids,dtype=torch.bool)
        memory=model.encode_prompt(ids,mask)
        x=model.decoder_start.view(1,1,-1).expand(ids.size(0),1,-1)
        caches=[None]*len(model.decoder)
        for position in range(int(max_new_tokens)):
            for layer_index, layer in enumerate(model.decoder):
                x,caches[layer_index]=layer.forward_step(
                    x,memory["evidence_memory"],memory["case_memory"],
                    memory["src_attention_mask"],caches[layer_index],position
                )
            logits=F.linear(model.decoder_final_norm(x)[:,-1],model.token_embedding.weight).float()
            if PAD_ID is not None and PAD_ID != EOS_ID: logits[:,PAD_ID]=-float("inf")
            if BOS_ID is not None and BOS_ID != EOS_ID: logits[:,BOS_ID]=-float("inf")

            if repetition_penalty and repetition_penalty > 1.0 and gen:
                seen=set(gen)
                idx=torch.tensor(sorted(seen),dtype=torch.long,device=logits.device)
                vals=logits[0,idx]
                logits[0,idx]=torch.where(vals < 0, vals*float(repetition_penalty), vals/float(repetition_penalty))

            n=int(no_repeat_ngram_size or 0)
            if n >= 2 and len(gen) >= n-1:
                prefix=tuple(gen[-(n-1):])
                banned=set()
                for j in range(len(gen)-n+1):
                    ng=gen[j:j+n]
                    if tuple(ng[:-1])==prefix:
                        banned.add(ng[-1])
                if EOS_ID is not None: banned.discard(EOS_ID)
                if banned:
                    bidx=torch.tensor(list(banned),dtype=torch.long,device=logits.device)
                    logits[0,bidx]=-float("inf")

            next_id=int(torch.argmax(logits,dim=-1).item())
            if EOS_ID is not None and next_id==EOS_ID: break
            gen.append(next_id)
            x=model.token_embedding(torch.tensor([next_id],dtype=torch.long,device=DEVICE)).unsqueeze(1)

    out=TOKENIZER.decode(gen,skip_special_tokens=True).strip()
    return re.sub(r"\s+"," ",out)

# Four compact candidates: original baseline plus increasingly strong anti-repetition settings.
DECODING_CANDIDATES=[
    {"name":"greedy_baseline","repetition_penalty":1.00,"no_repeat_ngram_size":0,"max_new_tokens":160},
    {"name":"rp112_ngram4","repetition_penalty":1.12,"no_repeat_ngram_size":4,"max_new_tokens":160},
    {"name":"rp118_ngram4","repetition_penalty":1.18,"no_repeat_ngram_size":4,"max_new_tokens":160},
    {"name":"rp112_ngram3","repetition_penalty":1.12,"no_repeat_ngram_size":3,"max_new_tokens":160},
]
VAL_TUNE_N=min(12,len(VAL_ENC))
val_tune=VAL_ENC[:VAL_TUNE_N]
val_refs=[VAL_RECORDS[i]["summary"] for i in range(VAL_TUNE_N)]
if VAL_TUNE_N < 8:
    raise RuntimeError(f"Only {VAL_TUNE_N} validation records available; at least 8 are required for decoding selection.")

try:
    from bert_score import score as e8_bertscore
except Exception:
    e8_bertscore=None
    try:
        print("Installing bert-score for validation-only scoring...")
        subprocess.run([sys.executable,"-m","pip","install","-q","bert-score"],check=True,timeout=240)
        from bert_score import score as e8_bertscore
    except Exception as exc:
        print("BERTScore unavailable; selection will fall back to ROUGE-L / ROUGE-1:",repr(exc)[:240])

validation_results=[]
validation_texts={}
for ci,cfg in enumerate(DECODING_CANDIDATES,1):
    preds=[]
    print(f"\nVALIDATION CANDIDATE {ci}/{len(DECODING_CANDIDATES)}: {cfg['name']}")
    for i,item in enumerate(val_tune,1):
        preds.append(generate_c_controlled(model,item,**{k:cfg[k] for k in ("max_new_tokens","repetition_penalty","no_repeat_ngram_size")}))
        if i % 4 == 0 or i == len(val_tune): print(f"  generated {i}/{len(val_tune)}")
    r1=float(np.mean([e8_rouge1(p,r) for p,r in zip(preds,val_refs)]))
    rl=float(np.mean([e8_rouge_l(p,r) for p,r in zip(preds,val_refs)]))
    rep=float(np.mean([e8_repetition_stats(p)[0] for p in preds]))
    rep4=float(np.mean([e8_repetition_stats(p)[1] for p in preds]))
    bs=None
    if e8_bertscore is not None:
        try:
            _,_,f1=e8_bertscore(preds,val_refs,model_type="roberta-large",lang="en",device=str(DEVICE),batch_size=2,verbose=False,rescale_with_baseline=False)
            bs=float(f1.mean().detach().cpu())
        except Exception as exc:
            print("  BERTScore unavailable for this run:",repr(exc)[:200])
            e8_bertscore=None
    row={**cfg,"validation_bertscore_f1":bs,"validation_rouge1_f1":r1,"validation_rougeL_f1":rl,
         "mean_duplicate_sentence_rate":rep,"mean_duplicate_4gram_rate":rep4}
    validation_results.append(row); validation_texts[cfg["name"]]=preds
    print("  METRICS:",json.dumps(row))

# Choose by the official primary metric when available. Among candidates within
# 0.008 BERTScore of the best, prefer stronger ROUGE-L, then fewer repeated sentences.
with_bs=[r for r in validation_results if r["validation_bertscore_f1"] is not None]
if with_bs:
    best_bs=max(r["validation_bertscore_f1"] for r in with_bs)
    near=[r for r in with_bs if r["validation_bertscore_f1"] >= best_bs-0.008]
    chosen=max(near,key=lambda r:(r["validation_rougeL_f1"],r["validation_rouge1_f1"],-r["mean_duplicate_sentence_rate"],-r["mean_duplicate_4gram_rate"]))
    selection_rule="BERTScore F1 primary; within 0.008 of best, tie-break by ROUGE-L, ROUGE-1, and lower sentence repetition"
else:
    chosen=max(validation_results,key=lambda r:(r["validation_rougeL_f1"],r["validation_rouge1_f1"],-r["mean_duplicate_sentence_rate"]))
    selection_rule="ROUGE-L primary fallback; tie-break by ROUGE-1 and lower sentence repetition"

CHOSEN_CONFIG={k:chosen[k] for k in ("name","repetition_penalty","no_repeat_ngram_size","max_new_tokens")}
print("\nSELECTED DECODING:",json.dumps(CHOSEN_CONFIG))
print("SELECTION RULE:",selection_rule)
(ROOT/"E8_decoding_validation_results.json").write_text(json.dumps({"selection_rule":selection_rule,"chosen":chosen,"candidates":validation_results},indent=2),encoding="utf-8")

In [ ]:

SUBMISSION_NAME="E8_C_LoRA_R16_RepetitionControlled"
STATE_PATH=E8_DIR/"generation_state.json"
saved={}
if STATE_PATH.exists():
    try:
        prior=json.loads(STATE_PATH.read_text(encoding="utf-8"))
        if prior.get("config_name")==CHOSEN_CONFIG["name"]:
            saved={str(k):str(v) for k,v in prior.get("predictions",{}).items()}
    except Exception:
        saved={}
expected_ids=[str(x["id"]) for x in TEST_ENC]
saved={k:v for k,v in saved.items() if k in set(expected_ids) and v.strip()}

for i,item in enumerate(TEST_ENC,1):
    rid=str(item["id"])
    if rid not in saved:
        saved[rid]=generate_c_controlled(model,item,
            max_new_tokens=CHOSEN_CONFIG["max_new_tokens"],
            repetition_penalty=CHOSEN_CONFIG["repetition_penalty"],
            no_repeat_ngram_size=CHOSEN_CONFIG["no_repeat_ngram_size"])
    if i%5==0 or i==len(TEST_ENC):
        tmp=STATE_PATH.with_suffix(".tmp")
        tmp.write_text(json.dumps({"config_name":CHOSEN_CONFIG["name"],"predictions":saved,"next_index":i},ensure_ascii=False),encoding="utf-8")
        os.replace(tmp,STATE_PATH)
        elapsed=(time.time()-E8_START_TIME)/60
        print(f"Test summaries {len(saved)}/{len(TEST_ENC)} | elapsed {elapsed:.1f} min")

prediction_rows=[{"id":x["id"],"summary":saved[str(x["id"])]} for x in TEST_ENC]
assert [str(x["id"]) for x in prediction_rows]==expected_ids,"Test ID/order mismatch"
assert len(prediction_rows)==len(TEST_RECORDS)
assert all(isinstance(x["summary"],str) and x["summary"].strip() for x in prediction_rows),"Empty summary"

portal_zip=Path("/kaggle/working/E8_C_LoRA_R16_REPETITION_CONTROLLED_PORTAL_READY.zip")
portal_json=Path("/kaggle/working/prediction.json")
portal_json.write_text(json.dumps(prediction_rows,ensure_ascii=False,indent=2),encoding="utf-8")
with zipfile.ZipFile(portal_zip,"w",compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(portal_json,arcname="prediction.json")
with zipfile.ZipFile(portal_zip,"r") as zf:
    assert zf.testzip() is None
    assert zf.namelist()==["prediction.json"],zf.namelist()
    check=json.loads(zf.read("prediction.json").decode("utf-8"))
assert [str(x["id"]) for x in check]==expected_ids
assert len(check)==100 or len(check)==len(TEST_RECORDS)

manifest={
    "experiment":SUBMISSION_NAME,
    "base_model":"Phase-1 Architecture C / DMLC-150",
    "adapter":"E7 rank-16 LoRA best_adapter.pt",
    "adapter_path":str(ADAPTER_PATH),
    "selected_decoding":CHOSEN_CONFIG,
    "selection_rule":selection_rule,
    "validation_results":validation_results,
    "test_examples":len(prediction_rows),
    "zip_path":str(portal_zip),
    "zip_root_contents":["prediction.json"],
    "elapsed_minutes":round((time.time()-E8_START_TIME)/60,2),
    "status":"validated_portal_ready"
}
(ROOT/"E8_final_manifest.json").write_text(json.dumps(manifest,indent=2),encoding="utf-8")
print("\n"+"="*72)
print("SUCCESS — E8 PORTAL-READY ZIP CREATED")
print(json.dumps(manifest,indent=2))
print("="*72)
try:
    from IPython.display import display,FileLink
    display(FileLink(str(portal_zip)))
except Exception:
    pass
print("Download from Kaggle Output: /kaggle/working/E8_C_LoRA_R16_REPETITION_CONTROLLED_PORTAL_READY.zip")